# import library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn #pip install scikit-learn
import scipy.stats as stats 
import csv
import requests
import statsmodels.api as sm
from statsmodels.formula.api import ols
import re

# option de visualisation output

In [ ]:
pd.set_option('display.max_columns', None) # affiche toutes les colonnes
pd.set_option('display.max_colwidth', None)  # Affiche les colonnes dans toute leur largeur
#pd.set_option('display.max_rows', None)# Affiche les lignes dans toute leur largeu

# Create df

In [ ]:
data2=pd.read_csv("data2.csv")

# Explore

In [ ]:
data2.head()

In [ ]:
data2.shape

In [ ]:
data2.info()

# Modelisation 

## Definir problematique : 

BUT ANTHICIPER LES BESOINS DE CONSOMMATIONS DES BATIMENTS,
tenter de prédire les émissions de CO2 et la consommation totale d’énergie de bâtiments non destinés à l’habitation

donc, on regarde les données structurales des batiments, pour predire, selon la structure du batiment (surface etage...),et son utlisation quel sera sa consommation

## Model RUN 1

### Preparation data avant 1ere Modélisation
#### Choix des variables

Labels Y : list model 1
Features X : equivalent GHGEmissionsIntensity per square foot

#### Transfo Dataframe en numerique : class_bat

In [ ]:
# one shot encoding on this col
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(handle_unknown='ignore')#creating instance of one-hot-encoder
transform = encoder.fit_transform(data2[["Classe_bat"]])

In [ ]:
transform.toarray()

In [ ]:
#perform one-hot encoding on 'team' column 
encoder_df = pd.DataFrame(transform.toarray(),dtype ="Int32")

In [ ]:
encoder_df.shape

In [ ]:
encoder_df.columns = encoder. get_feature_names_out(["Classe_bat"])

In [ ]:
encoder_df.columns

In [ ]:
encoder_df

In [ ]:
encoder_df.nunique()

In [ ]:
list_tempo =  ['CouncilDistrictCode',
 'Classe_Age',
 'NumberofBuildings',
 'NumberofFloors',
 'PropertyGFABuilding(s)',"GHGEmissionsIntensity" ]

In [ ]:
df_list_tempo = data2[list_tempo]

In [ ]:
df_list_tempo.shape

In [ ]:
df_list_tempo.info()

In [ ]:
df_list_tempo.dropna(subset ="GHGEmissionsIntensity",inplace = True)

In [ ]:
df_list_tempo.info()

In [ ]:
df_list_tempo.reset_index(inplace = True)
df_list_tempo.drop(columns = ["index"],inplace =True)

In [ ]:
df_list_tempo

In [ ]:
# Create a dataframe
#df_model1 = pd.concat([data2[list_tempo],encoder_df], axis=1)
df_model1 = df_list_tempo.join(encoder_df)

In [ ]:
df_model1

In [ ]:
X = df_model1 .drop (columns = "GHGEmissionsIntensity")
Y = df_model1["GHGEmissionsIntensity"]

In [ ]:
df_model1.isna().count()

In [ ]:
Y.head()

#### Normalistation des datas

In [ ]:
from sklearn import preprocessing

# pas centrage seulement variance de 1 : Z score Normalization
std_scale = preprocessing.StandardScaler().fit(X)
X_scale = std_scale.transform(X)
df_X_scale = pd.DataFrame(X_scale, columns=X.columns)
df_X_scale.head() 


In [ ]:
# normaliser tout entre 0 et 1 mieux que juste variance 
# Trop de varaibilite entre les features, plus visible, plus de perf pour la modelisation car plus petite valeur
norm1 = sklearn.preprocessing.normalize(X, norm='l2', axis=0, copy=True, return_norm=False) # axis = 0 means normalization amoung the col/features..better
x_norm = pd.DataFrame(norm1, columns = X.columns)
x_norm 

In [ ]:
# Another option MinMaxScaler : transform features by scaling each feature to a given range.
# Min-max is a scaling technique where values are rescaled and shifted so that they range between 0 and 1
scalerMinMax = preprocessing.MinMaxScaler() #default=(0, 1)
names = X.columns
d = scalerMinMax.fit_transform(X)
x_normMM = pd.DataFrame(d, columns=names)
x_normMM.head() 

In [ ]:
x_normMM["CouncilDistrictCode"].unique()
# output : OK

In [ ]:
x_norm["CouncilDistrictCode"].max()

Methode of normalization

Min-max normalization is preferred when data doesn’t follow Gaussian or normal distribution. It’s favored for normalizing algorithms that don’t follow any distribution, such as KNN and neural networks. Note that normalization is affected by outliers.

Log scaling is preferable if a dataset holds huge outliers.


### Creer un jeu de données test et train : Test train split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_norm,Y, test_size = 0.2, random_state = 1) #random state : ontrols how the data is shuffled before the split is implemented.

In [ ]:
x_train.shape, x_test.shape, y_train.shape, y_test.shape

In [ ]:
# testons si nos echantillons sont bien equilibres:
(((Y.value_counts())/Y.shape[0])*100).round(2)

In [ ]:
(((y_train.value_counts())/y_train.shape[0])*100).round(2)

### Test Model :
#### Regression lineaire Simple

#####  Regression lineaire classique : baseline de regression des moindres carres

In [ ]:
from sklearn import linear_model
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import make_scorer, r2_score
# On crée un modèle de régression linéaire
lr = linear_model.LinearRegression() # droite f(x)

In [ ]:
# On entraîne ce modèle sur les données d'entrainement
lr.fit(x_train,y_train) # fit trouver les param qui minimise les moindres carres pour avoir le meilleur coef
print(lr.coef_) # 1 coef par feature X

In [ ]:
# Save coef in a dataframe
df_coef =pd.DataFrame(lr.coef_, X.columns, columns = ['LR_classic'])
df_coef #High coef, means High impact for the prediction
#output : the highest coef are Number of Building and area, make sens for CO2 emission...

In [ ]:
# Resultats
print(lr.intercept_) # see the intercept (b or Beta0) of the regression equation

Prediction: score test et train
sklearn mean square error

Resultats Attendus : 
normalement score train jamais sup au test

-suarprentissage si train top et test nul
-sous aprr si train et test bof
donc augmenter feature ou model plus complex

In [ ]:
y_pred_lr_test = lr.predict(x_test)
y_pred_lr_train = lr.predict(x_train)

In [ ]:
# On récupère l'erreur de norme 2 sur le jeu de données test comme baseline
baseline_error_test = np.mean((y_pred_lr_test - y_test) ** 2) # moy des erreurs quadratiques entre donnee et prediction sur test, cad diff entre valeur obs et valeur predites 
baseline_error_train = np.mean((y_pred_lr_train - y_train) ** 2) # moy des erreurs quadratiques entre donnee et prediction sur training, cad diff entre valeur obs et valeur predites 

# Erreur quadratique = MSE: Mean Scare Error
print("Pour la régression linéaire basique : La MSE du training set est de : " + str(baseline_error_train.round(2)))
print("Pour la régression linéaire basique : La MSE du testing set est de : " + str(baseline_error_test.round(2)))

Interpretation of a train score > test score :

Imagine a test set consinting of observation that perfectly match equation you got from train set. In such situation test-set-MSE is 0.

When does it happen? When you're lucky and future data fit your model better than data on which it was derived.

Or when you model is weak (high MSE), so it's easy to get better data for it by chance.

Or when your test set is small. It's again quite easy to have great-matching observations by chance.

In [ ]:
# MSE via Sklearn
MSE_lr_test_sk = sklearn.metrics.mean_squared_error(y_test, y_pred_lr_test)
MSE_lr_train_sk = sklearn.metrics.mean_squared_error(y_train, y_pred_lr_train)

In [ ]:
MSE_lr_test_sk, MSE_lr_train_sk # Meme resultats

In [ ]:
# RMSE via Sklearn : Racine de MSE, vire le carre
RMSE_lr_test_sk = sklearn.metrics.mean_squared_error(y_test, y_pred_lr_test, squared= False)
RMSE_lr_train_sk = sklearn.metrics.mean_squared_error(y_train, y_pred_lr_train, squared= False)

In [ ]:
RMSE_lr_test_sk, RMSE_lr_train_sk 

In [ ]:
# R2 (coef de correlation)
r2_score(y_train, y_pred_lr_train)
print("Pour la régression linéaire basique : Le coef. de correlation R2 du testing set est de : " + str(r2_score(y_test, y_pred_lr_test)))


In [ ]:
# Visualisation perf du model1
plt.scatter(y_train, y_pred_lr_train)
plt.xlabel("Emission Co2 Prédites sur training set")
plt.ylabel("Emission Co2 réelles sur training set")
plt.xlim(0,20)
plt.ylim(0,7)
# OUTPUT PAS DE CORRELATION, DONC PAS TOP

In [ ]:
# Visualisation 
plt.scatter(y_test, y_pred_lr_test)
plt.xlabel("Emission Co2 Prédites sur jeu test")
plt.ylabel("Emission Co2 réelles sur jeu test")
# pas mieux sur testing set

Resultats:
our predicted values are NOT REALLY close to the actual values for the observations in the data set. 
(A perfectly straight diagonal line in this scatterplot would indicate that our model perfectly predicted the y-array values.)

Need improving :)

In [ ]:
# Another way to visually assess the performance of our model is to plot its residuals = the difference between the actual y-array values and the predicted y-array values.
plt.hist(y_test - y_pred_lr_test) #distribution difference between obs and prediction
# output : not normally distributed : not good :)

##### Dummy Regressor

In [ ]:
from sklearn import dummy
dum = dummy.DummyRegressor(strategy='mean')

In [ ]:

# Define Model
model1_dum = dummy.DummyRegressor().fit( x_train, y_train)
print("Dummy-R2 sur training set est de (0): "+str(model1_dum.score(x_train,y_train))+" : OK.")# output= 0

# Prédiction sur le jeu de test
y_pred_dum2 = model1_dum.predict(x_test)
print("Dummy-R2 sur testing set est de : "+str(model1_dum.score(x_test,y_test)))

In [ ]:
# Evaluate
print("MSE with Dummy Regressor sur le testing set: {:.2f}".format(sklearn.metrics.mean_squared_error(y_test, y_pred_dum2)) )
print("RMSE with Dummy Regressor : {:.2f}".format(np.sqrt(sklearn.metrics.mean_squared_error(y_test, y_pred_dum2)) ))

In [ ]:
y_pred_dum = model1_dum.predict(x_train)
print("MSE with Dummy Regressor sur le training set : {:.2f}".format(sklearn.metrics.mean_squared_error(y_train, y_pred_dum)) )

In [ ]:
# Visualize res (actual testing values vs predicted)
plt.scatter(y_test, y_pred_dum2)
plt.xlabel("Emission Co2 Prédites par Dummy")
plt.ylabel("Emission Co2 réelles")
plt.title("DummyRegressor method on testing set")


##### Ameliorer la regression lineraire : Trouver le meilleur alpha/lambda as hyperparametres :
ridge
lasso
gridsearch sklearn

###### Ridge/Tykhonov regularisation : permet de reduire l amplitude des coef d une regression lineaire et eviter le surapprentissage
pose une contrainte : lambda/ alpha

In [ ]:
# Coef lambda/alpha a tester
n_alphas = 200
alphas = np.logspace(-5, 5, n_alphas) # test different alpha, entre 10 puissance -5 et 10 puiss 5

In [ ]:
from sklearn.linear_model import Ridge
# Regression sous contrainte lambda, test pour chaque lambda/alpha
ridge = linear_model.Ridge()
coefs = []
errors = []
for a in alphas:
    ridge.set_params(alpha=a)
    ridge.fit(x_train, y_train)
    coefs.append(ridge.coef_)
    errors.append([baseline_error_test, np.mean((ridge.predict(x_test) - y_test) ** 2)])

In [ ]:
# MSE
MSE_lr_ridge = np.mean((ridge.predict(x_test) - y_test) ** 2)
print("Pour la régression linéaire ridge : La MSE du testing set est de : " + str(MSE_lr_ridge.round(2)))

In [ ]:
np.argmin(errors,axis = 0)

In [ ]:
len(errors)

In [ ]:
# np.argmin(errors[:,1])
coef_opt = coefs[155]
coef_opt

In [ ]:
# Visualisation de la difference entre (erreur quadratique : cest diff entre yobs et y predit (f(x)) au carree)
erreurs= np.array(errors)
plt.semilogx(alphas,erreurs[:,1]) # valeur de erreur en fonction de alpha
plt.semilogx(alphas, erreurs[:,0]) # baseline error, from linear regression (erreur quadratrique moy sur jeu test), ev
# output : il existe une valeur alpha/lambda qui ameliore la prediction (ameliore l erreur sur les données de test) 
# ici tous les alphas ameliorent

In [ ]:
np.argmin(erreurs[:,1]) # on trouve l indice de l erreur mini qui ameliore l erreur

In [ ]:
# on veut retrouver alpha pour l erreur min
alpha_opt = alphas[np.argmin(erreurs[:,1])]
alpha_opt

In [ ]:
# Chemin de regularisation
plt.semilogx(alphas, coefs)
plt.semilogx([alpha_opt, alpha_opt], [-4,4], 'r--')

In [ ]:
coef_opt = coefs[np.argmin(erreurs[:,1])]
coef_opt

##### Lasso : methode de selection de variables et de reduction de dimension supervise
remove celle norme 1 de B sup a t
region admissible +^- valeur absolue B

In [ ]:
from sklearn.linear_model import Lasso
# Regression sous contrainte lambda, test pour chaque lambda/alpha
lasso = linear_model.Lasso()

coefs2 = []
errors2 = []
for a in alphas:
    lasso.set_params(alpha=a)
    lasso.fit(x_train, y_train)
    coefs2.append(lasso.coef_)
    errors2.append([baseline_error_test, np.mean((lasso.predict(x_test) - y_test) ** 2)])

In [ ]:
# MSE
MSE_lr_lasso = np.mean((lasso.predict(x_test) - y_test) ** 2)
print("Pour la régression linéaire lasso : La MSE du testing set est de : " + str(MSE_lr_lasso.round(2)))


In [ ]:
# Visualisation de la difference entre (erreur quadratique : cest diff entre yobs et y predit (f(x)) au carree)
erreurs2= np.array(errors2)
plt.semilogx(alphas,erreurs2[:,1]) # valeur de erreur en fonction de alpha
plt.semilogx(alphas, erreurs2[:,0]) # baseline error, from linear regression (erreur quadratrique moy sur jeu test), ev
# output : il existe une valeur alpha/lambda qui ameliore la prediction (ameliore l erreur sur les données de test) 

In [ ]:
alpha_opt_lasso = alphas[np.argmin(erreurs2[:,1])]

# Chemin de regularisation
plt.semilogx(alphas, coefs2)
plt.semilogx([alpha_opt_lasso, alpha_opt_lasso], [-4,4], 'r--')

In [ ]:
# Chemin de regularisation
plt.semilogx(alphas, coefs2)
plt.xlim(.0001,.1)
plt.ylim(-.4,.2)
plt.semilogx([alpha_opt_lasso, alpha_opt_lasso], [-.5,.5], 'r--')

In [ ]:
alpha_opt_lasso 

In [ ]:
np.argmin(erreurs2[:,1])

In [ ]:
coef_opt2 = coefs2[np.argmin(erreurs2[:,1])]
coef_opt2

##### Grid search technique . recherche hyperparametre
test les models
GridSearchCV should be used to find the optimal parameters to train your final model.
Typically, you should run GridSearchCV
then look at the parameters that gave the model with the best score.
You should then take these parameters and train your final model on all of the data. 
It is important to note that if you have trained your final model on all of your data, you cannot test it. 

In [ ]:
from sklearn.model_selection import GridSearchCV

###### LR

In [ ]:
# Define params
param_linear_grid = {'fit_intercept':[True,False], 'normalize':[True,False]}

In [ ]:
grid = GridSearchCV(lr,param_linear_grid,cv=10)
grid.fit(x_train, y_train)

In [ ]:

print("MSE gridsearch with kf=10 and lr method: %.2f"
              % np.mean((grid.predict(x_test) - y_test) ** 2))
print ("r2 / variance : ", grid.best_score_)

##### Lasso

In [ ]:
# Define params
param_lasso_grid = {'fit_intercept':[True,False], 'normalize':[True,False],'alpha': alphas}
# [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100]

In [ ]:
grid_lasso = GridSearchCV(lasso, param_lasso_grid,scoring='neg_mean_absolute_error',cv =10,verbose=3) 
grid_lasso.fit(x_train, y_train)

In [ ]:
# Results
print("MSE gridsearch with kf=10 and lasso method: %.2f"
              % np.mean((grid_lasso.predict(x_test) - y_test) ** 2))
print ("r2 / variance : ", grid_lasso.best_score_)
print('Best Hyperparameters: %s' % grid_lasso.best_params_)

In [ ]:
df_res1_lasso_grid = pd.DataFrame(grid_lasso.cv_results_)
df_res1_lasso_grid["alpha"] = df_res1_lasso_grid.params.apply(lambda x: x["alpha"])


In [ ]:
df_res1_lasso_grid.columns

In [ ]:
fig = plt.figure(figsize=(5, 5))
ax = plt.subplot()
df_res1_lasso_grid.set_index("alpha")[["mean_test_score"]].plot(ax=ax)
ax.set_xlabel("alpha")
ax.set_ylabel("mean_test_score")
plt.xlim([0,1])
#plt.xscale('log')

###### Ridge

In [ ]:
grid_ridge = GridSearchCV(ridge, param_lasso_grid,scoring='neg_mean_absolute_error',cv =10,verbose=3) 
grid_ridge.fit(x_train, y_train)

In [ ]:
# Results
print("MSE gridsearch with kf=10 and lasso method: %.2f"
              % np.mean((grid_ridge.predict(x_test) - y_test) ** 2))
print ("r2 / variance : ", grid_ridge.best_score_)
print('Best Hyperparameters: %s' % grid_ridge.best_params_)

In [ ]:
df_res1_ridge_grid = pd.DataFrame(grid_ridge.cv_results_)
df_res1_ridge_grid["alpha"] = df_res1_ridge_grid.params.apply(lambda x: x["alpha"])

In [ ]:
fig = plt.figure(figsize=(5, 5))
ax = plt.subplot()
df_res1_ridge_grid.set_index("alpha")[["mean_test_score"]].plot(ax=ax)
ax.set_xlabel("alpha")
ax.set_ylabel("mean_test_score")
plt.xlim([0,2000])

##### Validation croisee pour choisir k fold
= méthode d’estimation de fiabilité d’un modèle fondée sur une technique d’échantillonnage.
choisir les meilleurs hyper-paramètres
but : optimiser l’erreur quadratique moyenne (Mean Squared Error, MSE)

In [ ]:
from sklearn.model_selection import KFold

In [ ]:
# Methode Kfold sklearn ( au lieu de faire un seuk train set, faire tour a tour)
 # K group = 10 pour nos 1500 datas
kf = KFold(n_splits=10)
for train, test in kf.split(df_model1):
    print("%s %s" % (train, test)) 

###### Cross Val score : MSE

In [ ]:
# How to estimate the accuray of a linear kernel (10 times)
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import make_scorer, r2_score
# return array of score (default = R2) of the estimator for each run
scores_lr = cross_val_score(lr, X, Y, cv=kf,scoring = make_scorer(mean_squared_error)) 
scores_lr # DONNE SCORE (MSE) A CHAQUE ITERATION


In [ ]:
print ("Le meilleur MSE, validation croisée LR : " + str ((min(scores_lr)).round(2)) )

In [ ]:
# Rigde
scores_ridge = cross_val_score(ridge, X, Y, cv=kf,scoring = make_scorer(mean_squared_error)) 
print ("Le meilleur MSE, validation croisée ridge : " + str ((min(scores_ridge)).round(2)) )

In [ ]:
# Lasso
scores_lasso = cross_val_score(lasso, X, Y, cv=kf,scoring = make_scorer(mean_squared_error) ) 
print ("Le meilleur MSE, validation croisée Lasso : " + str ((min(scores_lasso)).round(2)) )

In [ ]:
# report performance
print('MSE from lr, mean (sdt) : %.3f (%.3f)' % (np.mean(scores_lr), np.std(scores_lr)))

print('MSE from ridge, mean (sdt):  %.3f (%.3f)' % (np.mean(scores_ridge), np.std(scores_ridge)))

print('MSE from lasso, mean (sdt):  %.3f (%.3f)' % (np.mean(scores_lasso), np.std(scores_lasso)))

#### Methode à Noyau = transfo non lineaire en lineaire

##### Linear SVR (Support Vector Machine for linear regression)

In [ ]:
# Import library
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

###### SVR libsvm implementation 

In [ ]:
#lr_SVR2 = make_pipeline(SVR(C=1.0, epsilon=0.2)) #StandardScaler()
lr_SVR = sklearn.svm.SVR(kernel='linear')
# Entraine le model
lr_SVR.fit(x_train, y_train)

In [ ]:
# Check params
lr_SVR.get_params()

In [ ]:
# Prediction sur testing set
y_pred_SVR = lr_SVR.predict(x_test)

In [ ]:
# MSE via Sklearn
MSE_SVR = sklearn.metrics.mean_squared_error(y_test, y_pred_SVR)
print ("le coef de correlation de la SVR sur testing set est de : MSE_SVR = "+ str(MSE_SVR.round(2)))


In [ ]:
coef_SVR = lr_SVR.coef_
coef_SVR

###### SVR liblinear  implementation 

In [ ]:
# for large data set > 10k row, use linearSVR or SGDRegressor

LinearSVR = sklearn.svm.LinearSVR(epsilon=0.2)

In [ ]:
# Entraine le model
LinearSVR.fit(x_train, y_train)

In [ ]:
# Check params
LinearSVR.get_params()

In [ ]:
# Prediction sur testing set
y_pred_LinearSVR = LinearSVR.predict(x_test)

In [ ]:
# MSE via Sklearn
MSE_LinearSVR = sklearn.metrics.mean_squared_error(y_test, y_pred_LinearSVR)
print ("MSE_LinearSVR = "+ str(MSE_LinearSVR.round(2)))


In [ ]:
coef_LinearSVR = LinearSVR.coef_
coef_LinearSVR

##### kRR Regression ridge kernel trics : RGF /Gaussien
identical to support SVR
KRR uses squared error loss while support vector regression uses -insensitive loss, both combined with l2 regularization

In [ ]:
from sklearn import kernel_ridge
kRR_GRF = kernel_ridge.KernelRidge(
    alpha=1.0, # valeur par défaut 
    kernel='rbf', # noyau Gaussien
    gamma=0.01)   # valeur de 1/(2 * sigma**2)
                                     
# entraîner le classifieur sur le jeu d'entrainement
kRR_GRF.fit(x_train, y_train)


In [ ]:
# prédire sur le jeu de test
y_pred_kRR = kRR_GRF.predict(x_test)

In [ ]:
# calculer la MSE sur le jeu de test
MSE_kRR_GRF = sklearn.metrics.mean_squared_error(y_test, y_pred_kRR)
print ("MSE_kRR_RBF : {:.2f}".format(MSE_kRR_GRF))


In [ ]:
vector_kRR_GRF = kRR_GRF.dual_coef_
vector_kRR_GRF # Weight vector in kernel space

In [ ]:
# créer une figure
fig = plt.figure(figsize=(6, 6))

# Compter, pour chaque paire de valeurs (y, y') où y est un vrai score et y' le score prédit,
# le nombre de ces paires.
# Ce nombre sera utilisé pour modifier la taille des marqueurs correspondants 
# dans un nuage de points
sizes = {}
for (yt, yp) in zip(list(y_test), list(y_pred_kRR)):
    if (yt, yp) in sizes.keys():
        sizes[(yt, yp)] += 1
    else:
        sizes[(yt, yp)] = 1        
keys = sizes.keys()

# afficher les prédictions
plt.scatter([k[0] for k in keys], 
            [k[1] for k in keys], 
             s=[sizes[k] for k in keys], 
            label="gamma = 0.01: MSE = {:.2f}".format(MSE_kRR_GRF))

# étiqueter les axes et le graphique
plt.xlabel('Score observé', fontsize=16)
plt.ylabel(u'Score prédit', fontsize=16)
plt.title('kernel Ridge Regression-RBF', fontsize=16)

# limites des axes
plt.xlim([0, 10])

# afficher la légende
plt.legend(loc="lower right", fontsize=12)

In [ ]:
# Resultat extremement bof

###### Gridsearch amelioration

In [ ]:
# valeurs du paramètre C, cf alpha de regression ridge (n=200, entre log de -5 et log 5)
alpha_range = np.logspace(-2, 2, 5)
# valeurs du paramètre gamma
gamma_range = np.logspace(-2, 1, 4)

# grille de paramètres
param_grid = {'alpha': alpha_range, 'gamma': gamma_range, 'kernel': ['linear', 'poly', 'rbf']}

# score pour sélectionner le modèle optimal
score = 'neg_mean_squared_error'

# initialiser la validation croisée
grid_pred_kRR = sklearn.model_selection.GridSearchCV(
        kernel_ridge.KernelRidge(kernel='rbf'),
        param_grid,
        cv=10,
        scoring=score)

# exécuter la validation croisée sur le jeu d'entraînement
grid_pred_kRR.fit(x_train, y_train)


# prédire sur le jeu de test avec le modèle sélectionné 
y_test_pred_kRR_grid = grid_pred_kRR.predict(x_test)

# calculer la RMSE correspondante
rmse_kRR_Grid = np.sqrt(sklearn.metrics.mean_squared_error(y_test, y_test_pred_kRR_grid))

# calculer la MSE sur le jeu de test
MSE_kRR_GRF_Grid = sklearn.metrics.mean_squared_error(y_test, y_test_pred_kRR_grid)
print ("MSE_kRR_RBF_grid : {:.2f}".format(MSE_kRR_GRF_Grid))


##### Polynomial-kRR

In [ ]:
kRR_poly1 = sklearn.kernel_ridge.KernelRidge(
    alpha=100, # valeur par défaut 
    kernel='poly', # noyau polynomial de degre 1
    gamma=0.1,
    degree=1)
                                     
# entraîner sur le jeu d'entrainement
kRR_poly1.fit(x_train, y_train)

# prédire sur le jeu de test
y_pred_kRR_poly1 = kRR_poly1.predict(x_test)


# calculer la MSE sur le jeu de test
MSE_kRR_poly1 = sklearn.metrics.mean_squared_error(y_test, y_pred_kRR_poly1)
print ("MSE_kRR_poly1 : {:.4f}".format(MSE_kRR_poly1))


In [ ]:
vector_kRR_poly1 = kRR_poly1.dual_coef_
vector_kRR_poly1 # Weight vector in kernel space

In [ ]:
kRR_poly2 = sklearn.kernel_ridge.KernelRidge(
    alpha=100, # valeur par défaut 
    kernel='poly', # noyau polynomial de degre 2
    gamma=0.01,
    degree=2)
                                     
# entraîner sur le jeu d'entrainement
kRR_poly2.fit(x_train, y_train)

# prédire sur le jeu de test
y_pred_kRR_poly2 = kRR_poly2.predict(x_test)


# calculer la MSE sur le jeu de test
MSE_kRR_poly2 = sklearn.metrics.mean_squared_error(y_test, y_pred_kRR_poly2)
print ("MSE_kRR_poly2 : {:.4f}".format(MSE_kRR_poly2))

In [ ]:
kRR_poly3 = sklearn.kernel_ridge.KernelRidge(
    alpha=100, # valeur par défaut 
    kernel='poly', # noyau polynomial de degre 3
    gamma=0.01,
    degree=3)
                                     
# entraîner sur le jeu d'entrainement
kRR_poly3.fit(x_train, y_train)

# prédire sur le jeu de test
y_pred_kRR_poly3 = kRR_poly3.predict(x_test)


# calculer la MSE sur le jeu de test
MSE_kRR_poly3 = sklearn.metrics.mean_squared_error(y_test, y_pred_kRR_poly3)
print ("MSE_kRR_poly3 : {:.4f}".format(MSE_kRR_poly3))

In [ ]:
deglist=[1,2,3,4,5,6,7,8,9,10,20]
MSE_krr_poly_deg=[]
for deg in deglist:
    kRR_poly5 = sklearn.kernel_ridge.KernelRidge(
        alpha=100, # valeur par défaut 
        kernel='poly', # noyau polynomial de degre 1
        gamma=0.01,
        degree=deg)                                 
    # entraîner sur le jeu d'entrainement
    kRR_poly5.fit(x_train, y_train)
    # prédire sur le jeu de test
    y_pred_kRR_poly5 = kRR_poly5.predict(x_test)
    # calculer la MSE sur le jeu de test
    MSE_krr_poly_deg.append(sklearn.metrics.mean_squared_error(y_test, y_pred_kRR_poly5))

plt.plot(deglist, MSE_krr_poly_deg, 'ob')
plt.ylim([4.27, 4.28])
plt.xlabel('degree du polynome')
plt.ylabel('MSE kernel polynomial')

#### Methode ensembliste
##### SGDRegressor 
GBR = GradientBoostingRegressor()

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
#sklearn.linear_model.SGDRegressor(loss='squared_error', *, penalty='l2', alpha=0.0001, l1_ratio=0.15, fit_intercept=True, max_iter=1000, tol=0.001, shuffle=True, verbose=0, epsilon=0.1, random_state=None, learning_rate='invscaling', eta0=0.01, power_t=0.25, early_stopping=False, validation_fraction=0.1, n_iter_no_change=5, warm_start=False, average=False)
GBR = sklearn.ensemble.GradientBoostingRegressor(random_state=1) #default criteirm friedman mse , mse upgraded
GBR.fit(x_train, y_train)

In [ ]:
GBR_pred = GBR.predict(x_test)

In [ ]:
GBR.score(x_test,y_test) #R2

In [ ]:
GBR.get_params()

In [ ]:
# MSE via Sklearn
MSE_GBR = sklearn.metrics.mean_squared_error(y_test, GBR_pred)
print ("MSE_GBR : {:.2f}".format(MSE_GBR))

In [ ]:
#FEATURE IMPORTANCE
GBR.feature_importances_

In [ ]:
# Gradient boosting
crossvalidation=KFold(n_splits=10,shuffle=True,random_state=1)
search_grid={'n_estimators':[100],'learning_rate':[.001,0.01,.1],'max_depth':[1,2,4],'subsample':[.5,.75,1],'random_state':[0,1]}
search=GridSearchCV(estimator=GBR,param_grid=search_grid,scoring='neg_mean_squared_error',n_jobs=1,cv=crossvalidation)

search.fit(x_train,y_train)



In [ ]:
search.best_score_, search.best_params_

In [ ]:
scores_GBR = cross_val_score(search, x_test,y_test, cv=kf,scoring = make_scorer(mean_squared_error)) 


In [ ]:
scores_GBR,np.mean(scores_GBR), np.min(scores_GBR)

In [ ]:
GBR_Pred_grid = search.predict(x_test)
MSE_GBR_grid = sklearn.metrics.mean_squared_error(y_test, GBR_Pred_grid)
print ("MSE_GBR : {:.2f}".format(MSE_GBR_grid))

##### RandomForest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# Etablissement Model
RFR = RandomForestRegressor(max_depth=2, random_state=0)

# Entrainement
RFR.fit(x_train, y_train)

# Predicton
RFR_Pred = RFR.predict(x_test)

In [ ]:
RFR.get_params()

In [ ]:
RFR.apply(x_test) #return leaf indice

In [ ]:
RFR.feature_importances_

In [ ]:
# MSE via Sklearn
MSE_RFR = sklearn.metrics.mean_squared_error(y_test, RFR_Pred)
print ("MSE_RFR : {:.2f}".format(MSE_RFR))

##### Amelioration des modeles : gridsearch

In [ ]:
# RFR
param_grid = {
    'bootstrap': [True],
    'max_depth': [80, 90, 100, 110],
    'max_features': [2, 3,4,5],
    'min_samples_leaf': [3, 4, 5],
    'min_samples_split': [8, 10, 12],
    'n_estimators': [100]
}

rf = RandomForestRegressor()
grid_search_rf = GridSearchCV(estimator = rf, param_grid = param_grid, 
                          scoring='neg_mean_squared_error',n_jobs=1,cv=crossvalidation, verbose = 2)
grid_search_rf.fit(x_train, y_train)

In [ ]:
grid_search_rf.best_score_, grid_search_rf.best_params_

In [ ]:
scores_RFR = cross_val_score(rf, x_test,y_test, cv=crossvalidation,scoring = make_scorer(mean_squared_error)) 
scores_RFR,np.mean(scores_RFR), np.min(scores_RFR)

In [ ]:
RFR_Pred_grid = grid_search_rf.predict(x_test)
MSE_RFR_grid = sklearn.metrics.mean_squared_error(y_test, RFR_Pred_grid)
print ("MSE_RFR : {:.2f}".format(MSE_RFR_grid))

In [ ]:
parameters = {'C': [0.1, 1, 10, 100, 1000],  
              'gamma': [1, 0.1, 0.01, 0.001, 0.0001,'auto'], 
              'kernel': ['linear', 'poly', 'rbf', 'sigmoid']} 

grid_pred_multi = sklearn.model_selection.GridSearchCV(
        kernel_ridge.KernelRidge(kernel='rbf'),
        param_grid,
        cv=10,
        scoring=score)


Svm = GridSearchCV(grid_pred_multi, param_grid=parameters, cv=kf,verbose=10)

In [ ]:
# valeurs du paramètre C, cf alpha de regression ridge (n=200, entre log de -5 et log 5)
alpha_range = np.logspace(-2, 2, 5)
# valeurs du paramètre gamma
gamma_range = np.logspace(-2, 1, 4)

# grille de paramètres
param_grid = {'alpha': alpha_range, 'gamma': gamma_range, 'kernel': ['linear', 'poly', 'rbf']}

# score pour sélectionner le modèle optimal
score = 'neg_mean_squared_error'

# initialiser la validation croisée
grid_pred_poly = sklearn.model_selection.GridSearchCV(
        kernel_ridge.KernelRidge(kernel=param_grid),
        param_grid,
        cv=10,
        scoring=score)

# exécuter la validation croisée sur le jeu d'entraînement
grid_pred_poly.fit(x_train, y_train)


# prédire sur le jeu de test avec le modèle sélectionné 
y_test_pred_poly_grid = grid_pred_poly.predict(x_test)

# calculer la RMSE correspondante
rmse_kRR_poly1_Grid = np.sqrt(sklearn.metrics.mean_squared_error(y_test, y_test_pred_poly_grid))

# calculer la MSE sur le jeu de test
MSE_kRR_poly_Grid = sklearn.metrics.mean_squared_error(y_test, y_test_pred_poly_grid)
print ("MSE_kRR_RBF_grid : {:.2f}".format(MSE_kRR_poly_Grid))


### Feature importance SHAP : SHapley Additive exPlanations
alternative to permutation feature importance
gbr svr only

In [ ]:
import shap
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# SVR LR
vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()

explainer_LinearSVR1 = shap.Explainer(LinearSVR, x_train, feature_names=x_train.columns)
shap_values1_LinearSVR = explainer_LinearSVR1(x_test)

In [ ]:
shap.plots.beeswarm(shap_values1_LinearSVR)

In [ ]:
shap.summary_plot(shap_values1_LinearSVR,features=x_train, feature_names=x_train.columns,plot_type='bar')

In [ ]:
# GBR
explainer_GBR1 = shap.Explainer(GBR, x_train, feature_names=x_train.columns)
shap_values_GBR1 = explainer_GBR1(x_test)
shap.plots.beeswarm(shap_values_GBR1)


In [ ]:
shap.summary_plot(shap_values_GBR1,features=x_test, feature_names=x_test.columns,plot_type='bar')

## Model RUN 1 test log 

### Preparation data avant 1ere Modélisation
#### Choix des variables . permutation en log

In [ ]:
data2["LogPropertyGFABuilding"] = np.log(data2["PropertyGFABuilding(s)"]) 
data2["LogLargestPropertyUseTypeGFA"] = np.log(data2["LargestPropertyUseTypeGFA"])

In [ ]:
list_col_struct_model12 = [
 'CouncilDistrictCode',
 'Classe_Age',
 'NumberofBuildings',
 'NumberofFloors',
 "LogPropertyGFABuilding","GHGEmissionsIntensity" ]

Labels Y : list model 1
Features X : equivalent GHGEmissionsIntensity per square foot

#### Transfo Dataframe en numerique : class_bat

In [ ]:
df_list_tempo2 = data2[list_col_struct_model12]

In [ ]:
df_list_tempo2.shape

In [ ]:
df_list_tempo2.info()

In [ ]:
# Remove Nan
df_list_tempo2.dropna(subset ="GHGEmissionsIntensity",inplace = True)

In [ ]:
df_list_tempo2.info() # verif si ok

In [ ]:
df_list_tempo2.reset_index(inplace = True)
df_list_tempo2.drop(columns = ["index"],inplace =True)

In [ ]:
df_list_tempo2

In [ ]:
# Create a dataframe
df_model12 = df_list_tempo2.join(encoder_df)

In [ ]:
df_model12

In [ ]:
X12 = df_model12.drop (columns = "GHGEmissionsIntensity")

In [ ]:
X12.isna().count()

In [ ]:
Y.head()

#### Normalistation des datas

In [ ]:
# normaliser tout entre 0 et 1 mieux que juste variance 
# Trop de varaibilite entre les features, plus visible, plus de perf pour la modelisation car plus petite valeur
norm12 = sklearn.preprocessing.normalize(X12, norm='l2', axis=0, copy=True, return_norm=False) # axis = 0 means normalization amoung the col/features..better
x12_norm = pd.DataFrame(norm12, columns = X12.columns)
x12_norm 

### Creer un jeu de données test et train : Test train split

In [ ]:
x12_train, x12_test = train_test_split(x12_norm, test_size = 0.2, random_state = 1) #random state : ontrols how the data is shuffled before the split is implemented.

In [ ]:
x12_train.shape, x12_test.shape, y_train.shape, y_test.shape

In [ ]:
# testons si nos echantillons sont bien equilibres:
(((Y.value_counts())/Y.shape[0])*100).round(2)

### Test Model :
#### Regression lineaire Simple

#####  Regression lineaire classique : baseline de regression des moindres carres

In [ ]:
# On entraîne ce modèle sur les données d'entrainement
lr.fit(x12_train,y_train) # fit trouver les param qui minimise les moindres carres pour avoir le meilleur coef
print(lr.coef_) # 1 coef par feature X

In [ ]:
# Save coef in a dataframe
df_coef_lr12 =pd.DataFrame(lr.coef_, X.columns, columns = ['Coeff'])
df_coef_lr12 #High coef, means High impact for the prediction
#output : the highest coef are Number of Building and area, make sens for CO2 emission...

In [ ]:
# Resultats
print(lr.intercept_) # see the intercept (b or Beta0) of the regression equation

Prediction: score test et train
sklearn mean square error

Resultats Attendus : 
normalement score train jamais sup au test

-suarprentissage si train top et test nul
-sous aprr si train et test bof
donc augmenter feature ou model plus complex

In [ ]:
y_pred_lr_test12 = lr.predict(x12_test)
y_pred_lr_train12 = lr.predict(x12_train)

In [ ]:
# On récupère l'erreur de norme 2 sur le jeu de données test comme baseline
baseline_error_test12 = np.mean((y_pred_lr_test12 - y_test) ** 2) # moy des erreurs quadratiques entre donnee et prediction sur test, cad diff entre valeur obs et valeur predites 
baseline_error_train12 = np.mean((y_pred_lr_train12 - y_train) ** 2) # moy des erreurs quadratiques entre donnee et prediction sur training, cad diff entre valeur obs et valeur predites 

# Erreur quadratique = MSE: Mean Scare Error
print("Pour la régression linéaire basique : La MSE du training set est de : " + str(baseline_error_train12.round(2)))
print("Pour la régression linéaire basique : La MSE du testing set est de : " + str(baseline_error_test12.round(2)))

In [ ]:
# MSE via Sklearn
MSE_lr_test12_sk = sklearn.metrics.mean_squared_error(y_test, y_pred_lr_test12)
MSE_lr_train12_sk = sklearn.metrics.mean_squared_error(y_train, y_pred_lr_train12)

In [ ]:
MSE_lr_test12_sk, MSE_lr_train12_sk # Meme resultats

In [ ]:
# RMSE via Sklearn : Racine de MSE, vire le carre
RMSE_lr_test12_sk = sklearn.metrics.mean_squared_error(y_test, y_pred_lr_test12, squared= False)
RMSE_lr_train12_sk = sklearn.metrics.mean_squared_error(y_train, y_pred_lr_train12, squared= False)

In [ ]:
RMSE_lr_test12_sk, RMSE_lr_train12_sk 

##### Dummy Regressor

In [ ]:
# Define Model
model12_dum = dummy.DummyRegressor().fit(x12_train, y_train)
print("Dummy-R2 sur training set est de (0): "+str(model12_dum.score(x12_train,y_train))+" : OK.")# output= 0

# Prédiction sur le jeu de test
y_pred12_dum = model12_dum.predict(x12_test)
print("Dummy-R2 sur testing set est de : "+str(model12_dum.score(x12_test,y_test)))

In [ ]:
# Evaluate
print("MSE with Dummy Regressor : {:.2f}".format(sklearn.metrics.mean_squared_error(y_test, y_pred12_dum)) )
print("RMSE with Dummy Regressor : {:.2f}".format(np.sqrt(sklearn.metrics.mean_squared_error(y_test, y_pred12_dum)) ))

##### Ameliorer la regression lineraire : Trouver le meilleur alpha/lambda as hyperparametres :
ridge
lasso
gridsearch sklearn

###### Ridge/Tykhonov regularisation : permet de reduire l amplitude des coef d une regression lineaire et eviter le surapprentissage
pose une contrainte : lambda/ alpha

In [ ]:
# Coef lambda/alpha a tester 200
coefs12 = []
errors12 = []
for a in alphas:
    ridge.set_params(alpha=a)
    ridge.fit(x12_train, y_train)
    coefs12.append(ridge.coef_)
    errors12.append([baseline_error_test12, np.mean((ridge.predict(x12_test) - y_test) ** 2)])

In [ ]:
# MSE
MSE12_lr_ridge = np.mean((ridge.predict(x12_test) - y_test) ** 2)
print("Pour la régression linéaire ridge : La MSE du testing set est de : " + str(MSE12_lr_ridge.round(2)))

In [ ]:
np.argmin(errors12,axis = 0)

In [ ]:
len(errors12)

In [ ]:
# np.argmin(errors[:,1])
coef12_opt = coefs12[155]
coef12_opt

In [ ]:
# Visualisation de la difference entre (erreur quadratique : cest diff entre yobs et y predit (f(x)) au carree)
erreurs12= np.array(errors12)
plt.semilogx(alphas,erreurs12[:,1]) # valeur de erreur en fonction de alpha
plt.semilogx(alphas, erreurs12[:,0]) # baseline error, from linear regression (erreur quadratrique moy sur jeu test), ev
# output : il existe une valeur alpha/lambda qui ameliore la prediction (ameliore l erreur sur les données de test) 
# ici tous les alphas ameliorent

In [ ]:
# on veut retrouver alpha pour l erreur min
alpha_opt12 = alphas[np.argmin(erreurs12[:,1])]
alpha_opt12

In [ ]:
# Chemin de regularisation
plt.semilogx(alphas, coefs12)
plt.semilogx([alpha_opt12, alpha_opt12], [-4,4], 'r--')

In [ ]:
coef12_opt = coefs12[np.argmin(erreurs12[:,1])]
coef12_opt

##### Lasso : methode de selection de variables et de reduction de dimension supervise
remove celle norme 1 de B sup a t
region admissible +^- valeur absolue B

In [ ]:
coefs212 = []
errors212 = []
for a in alphas:
    lasso.set_params(alpha=a)
    lasso.fit(x12_train, y_train)
    coefs212.append(lasso.coef_)
    errors212.append([baseline_error_test12, np.mean((lasso.predict(x12_test) - y_test) ** 2)])

In [ ]:
# MSE
MSE12_lr_lasso = np.mean((lasso.predict(x12_test) - y_test) ** 2)
print("Pour la régression linéaire lasso : La MSE du testing set est de : " + str(MSE12_lr_lasso.round(2)))


In [ ]:
# Visualisation de la difference entre (erreur quadratique : cest diff entre yobs et y predit (f(x)) au carree)
erreurs212= np.array(errors212)
plt.semilogx(alphas,erreurs212[:,1]) # valeur de erreur en fonction de alpha
plt.semilogx(alphas, erreurs212[:,0]) # baseline error, from linear regression (erreur quadratrique moy sur jeu test), ev
# output : il existe une valeur alpha/lambda qui ameliore la prediction (ameliore l erreur sur les données de test) 

In [ ]:
alpha_opt12_lasso = alphas[np.argmin(erreurs212[:,1])]

# Chemin de regularisation
plt.semilogx(alphas, coefs212)
plt.semilogx([alpha_opt12_lasso, alpha_opt12_lasso], [-4,4], 'r--')

In [ ]:
# Chemin de regularisation
plt.semilogx(alphas, coefs212)
plt.xlim(.0001,.1)
plt.ylim(-.4,.2)
plt.semilogx([alpha_opt12_lasso, alpha_opt12_lasso], [-.5,.5], 'r--')

In [ ]:
alpha_opt12_lasso 

In [ ]:
np.argmin(erreurs212[:,1])

In [ ]:
coef_opt212 = coefs212[np.argmin(erreurs212[:,1])]
coef_opt212

###### LR

In [ ]:
# grid = GridSearchCV(lr,param_linear_grid,cv=10)
grid.fit(x12_train, y_train)

In [ ]:
print("MSE gridsearch with kf=10 and lr method: %.2f"
              % np.mean((grid.predict(x12_test) - y_test) ** 2))
print ("r2 / variance : ", grid.best_score_)

###### Lasso

In [ ]:
grid_lasso.fit(x12_train, y_train)

In [ ]:
# Results
print("MSE gridsearch with kf=10 and lasso method: %.2f"
              % np.mean((grid_lasso.predict(x12_test) - y_test) ** 2))
print ("r2 / variance : ", grid_lasso.best_score_)
print('Best Hyperparameters: %s' % grid_lasso.best_params_)

In [ ]:
df_res12_lasso_grid = pd.DataFrame(grid_lasso.cv_results_)
df_res12_lasso_grid["alpha"] = df_res12_lasso_grid.params.apply(lambda x: x["alpha"])

In [ ]:
df_res12_lasso_grid.columns

###### Ridge

In [ ]:
grid_ridge.fit(x12_train, y_train)

In [ ]:
# Results
print("MSE gridsearch with kf=10 and lasso method: %.2f"
              % np.mean((grid_ridge.predict(x12_test) - y_test) ** 2))
print ("r2 / variance : ", grid_ridge.best_score_)
print('Best Hyperparameters: %s' % grid_ridge.best_params_)

In [ ]:
df_res12_ridge_grid = pd.DataFrame(grid_ridge.cv_results_)
df_res12_ridge_grid["alpha"] = df_res12_ridge_grid.params.apply(lambda x: x["alpha"])

##### Validation croisee pour choisir k fold

###### Cross Val score : MSE

In [ ]:

# return array of score (default = R2) of the estimator for each run
scores12_lr = cross_val_score(lr, X12, Y, cv=kf,scoring = make_scorer(mean_squared_error)) 
scores12_lr # DONNE SCORE (MSE) A CHAQUE ITERATION


In [ ]:
print ("Le meilleur MSE, validation croisée LR : " + str ((min(scores12_lr)).round(2)) )

In [ ]:
# Rigde
scores12_ridge = cross_val_score(ridge, X12, Y, cv=kf,scoring = make_scorer(mean_squared_error)) 
print ("Le meilleur MSE, validation croisée LR : " + str ((min(scores_ridge)).round(2)) )

In [ ]:
# Lasso
scores12_lasso = cross_val_score(lasso, X12, Y, cv=kf,scoring = make_scorer(mean_squared_error) ) 
print ("Le meilleur MSE, validation croisée LR : " + str ((min(scores_lasso)).round(2)) )

In [ ]:
# report performance
print('MSE from lr, mean (sdt) : %.3f (%.3f)' % (np.mean(scores12_lr), np.std(scores12_lr)))

print('MSE from ridge, mean (sdt):  %.3f (%.3f)' % (np.mean(scores12_ridge), np.std(scores12_ridge)))

print('MSE from lasso, mean (sdt):  %.3f (%.3f)' % (np.mean(scores12_lasso), np.std(scores12_lasso)))

#### Test Autre Model: Methode à Noyau =
transfo non lineaire en lineaire

##### Linear SVR (Support Vector Machine for linear regression)

###### SVR libsvm implementation 

In [ ]:
#lr_SVR2 = make_pipeline(SVR(C=1.0, epsilon=0.2)) #StandardScaler()
# lr_SVR = sklearn.svm.SVR(kernel='linear')
# Entraine le model
lr_SVR.fit(x12_train, y_train)

In [ ]:
# Prediction sur testing set
y_pred12_SVR = lr_SVR.predict(x12_test)

In [ ]:
# MSE via Sklearn
MSE12_SVR = sklearn.metrics.mean_squared_error(y_test, y_pred12_SVR)
print ("le coef de correlation de la SVR sur testing set est de : MSE_SVR = "+ str(MSE12_SVR.round(2)))


In [ ]:
coef12_SVR = lr_SVR.coef_
coef12_SVR

###### SVR liblinear  implementation 

In [ ]:
# Entraine le model
LinearSVR.fit(x12_train, y_train)

In [ ]:
# Prediction sur testing set
y_pred12_LinearSVR = LinearSVR.predict(x12_test)

In [ ]:
# MSE via Sklearn
MSE12_LinearSVR = sklearn.metrics.mean_squared_error(y_test, y_pred12_LinearSVR)
print ("MSE_LinearSVR = "+ str(MSE12_LinearSVR.round(2)))


In [ ]:
coef12_LinearSVR = LinearSVR.coef_
coef12_LinearSVR

##### kRR Regression ridge kernel trics : RGF /Gaussien
identical to support SVR
KRR uses squared error loss while support vector regression uses -insensitive loss, both combined with l2 regularization

In [ ]:
# Gaussien          
# entraîner  sur le jeu d'entrainement
kRR_GRF.fit(x12_train, y_train)


In [ ]:
# prédire sur le jeu de test
y_pred12_kRR = kRR_GRF.predict(x12_test)

In [ ]:
# calculer la MSE sur le jeu de test
MSE12_kRR_GRF = sklearn.metrics.mean_squared_error(y_test, y_pred_kRR)
print ("MSE_kRR_RBF : {:.2f}".format(MSE12_kRR_GRF))


In [ ]:
vector12_kRR_GRF = kRR_GRF.dual_coef_
vector12_kRR_GRF # Weight vector in kernel space

###### Gridsearch amelioration

In [ ]:
# exécuter la validation croisée sur le jeu d'entraînement
grid_pred_kRR.fit(x12_train, y_train)

# prédire sur le jeu de test avec le modèle sélectionné 
y_test12_pred_kRR_grid = grid_pred_kRR.predict(x12_test)

# calculer la RMSE correspondante
rmse12_kRR_Grid = np.sqrt(sklearn.metrics.mean_squared_error(y_test, y_test12_pred_kRR_grid))

# calculer la MSE sur le jeu de test
MSE12_kRR_GRF_Grid = sklearn.metrics.mean_squared_error(y_test, y_test12_pred_kRR_grid)
print ("MSE_kRR_RBF_grid : {:.2f}".format(MSE12_kRR_GRF_Grid))


##### Polynomial-kRR

In [ ]:
# entraîner sur le jeu d'entrainement
kRR_poly1.fit(x12_train, y_train)

# prédire sur le jeu de test
y_pred12_kRR_poly1 = kRR_poly1.predict(x12_test)


# calculer la MSE sur le jeu de test
MSE12_kRR_poly1 = sklearn.metrics.mean_squared_error(y_test, y_pred12_kRR_poly1)
print ("MSE_kRR_poly1 : {:.4f}".format(MSE12_kRR_poly1))

In [ ]:
vector12_kRR_poly1 = kRR_poly1.dual_coef_
vector12_kRR_poly1 # Weight vector in kernel space

In [ ]:
# entraîner sur le jeu d'entrainement
kRR_poly2.fit(x12_train, y_train)

# prédire sur le jeu de test
y_pred12_kRR_poly2 = kRR_poly2.predict(x12_test)


# calculer la MSE sur le jeu de test
MSE12_kRR_poly2 = sklearn.metrics.mean_squared_error(y_test, y_pred12_kRR_poly2)
print ("MSE_kRR_poly2 : {:.4f}".format(MSE12_kRR_poly2))

In [ ]:
# entraîner sur le jeu d'entrainement
kRR_poly3.fit(x12_train, y_train)

# prédire sur le jeu de test
y_pred12_kRR_poly3 = kRR_poly3.predict(x12_test)


# calculer la MSE sur le jeu de test
MSE12_kRR_poly3 = sklearn.metrics.mean_squared_error(y_test, y_pred12_kRR_poly3)
print ("MSE_kRR_poly3 : {:.4f}".format(MSE12_kRR_poly3))

In [ ]:
# deglist=[1,2,3,4,5,6,7,8,9,10,20]
MSE12_krr_poly_deg=[]
for deg in deglist:                               
    # entraîner sur le jeu d'entrainement
    kRR_poly5.fit(x12_train, y_train)
    # prédire sur le jeu de test
    y_pred12_kRR_poly5 = kRR_poly5.predict(x12_test)
    # calculer la MSE sur le jeu de test
    MSE12_krr_poly_deg.append(sklearn.metrics.mean_squared_error(y_test, y_pred12_kRR_poly5))

plt.plot(deglist, MSE12_krr_poly_deg, 'ob')
plt.ylim([4.27, 4.28])
plt.xlabel('degree du polynome')
plt.ylabel('MSE kernel polynomial')

#### METHODE ENSEMBLISTE
##### GDRegressor : GBR

In [ ]:
#sklearn.linear_model.SGDRegressor(loss='squared_error', *, penalty='l2', alpha=0.0001, l1_ratio=0.15, fit_intercept=True, max_iter=1000, tol=0.001, shuffle=True, verbose=0, epsilon=0.1, random_state=None, learning_rate='invscaling', eta0=0.01, power_t=0.25, early_stopping=False, validation_fraction=0.1, n_iter_no_change=5, warm_start=False, average=False)
# GBR = sklearn.ensemble.GradientBoostingRegressor(random_state=0) #default criteirm friedman mse , mse upgraded
GBR.fit(x12_train, y_train)

In [ ]:
GBR_pred12 = GBR.predict(x12_test)

In [ ]:
GBR.score(x12_test,y_test) #R2

In [ ]:
# MSE via Sklearn
MSE12_GBR = sklearn.metrics.mean_squared_error(y_test, GBR_pred12)
print ("MSE12_GBR : {:.2f}".format(MSE12_GBR))

In [ ]:
#FEATURE IMPORTANCE
GBR.feature_importances_

In [ ]:
# Gradient boosting
#crossvalidation=KFold(n_splits=10,shuffle=True,random_state=1)
#search_grid={'n_estimators':[100],'learning_rate':[.001,0.01,.1],'max_depth':[1,2,4],'subsample':[.5,.75,1],'random_state':[1]}
#search=GridSearchCV(estimator=GBR,param_grid=search_grid,scoring='neg_mean_squared_error',n_jobs=1,cv=crossvalidation)

search.fit(x12_train,y_train)



In [ ]:
search.best_score_, search.best_params_

In [ ]:
scores_GBR12 = cross_val_score(search, x12_test,y_test, cv=kf,scoring = make_scorer(mean_squared_error)) 


In [ ]:
scores_GBR12,np.mean(scores_GBR12), np.min(scores_GBR12)

In [ ]:
GBR_Pred_grid12 = search.predict(x12_test)
MSE12_GBR_grid = sklearn.metrics.mean_squared_error(y_test, GBR_Pred_grid12)
print ("MSE_GBR : {:.2f}".format(MSE12_GBR_grid))

##### RandomForest

In [ ]:
# Etablissement Model
# RFR = RandomForestRegressor(max_depth=2, random_state=0)

# Entrainement
RFR.fit(x12_train, y_train)

# Predicton
RFR_Pred12 = RFR.predict(x12_test)

In [ ]:
RFR.apply(x12_test) #return leaf indice

In [ ]:
RFR.feature_importances_

In [ ]:
# MSE via Sklearn
MSE12_RFR = sklearn.metrics.mean_squared_error(y_test, RFR_Pred12)
print ("MSE_RFR : {:.2f}".format(MSE12_RFR))

###### gridsearch

In [ ]:
# RFR
grid_search_rf.fit(x12_train, y_train)

In [ ]:
grid_search_rf.best_score_, grid_search_rf.best_params_

In [ ]:
scores_RFR12 = cross_val_score(rf, x12_test,y_test, cv=crossvalidation,scoring = make_scorer(mean_squared_error)) 
scores_RFR12,np.mean(scores_RFR12), np.min(scores_RFR12)

In [ ]:
RFR_Pred12_grid = grid_search_rf.predict(x12_test)
MSE12_RFR_grid = sklearn.metrics.mean_squared_error(y_test, RFR_Pred12_grid)
print ("MSE_RFR: {:.2f}".format(MSE12_RFR_grid))

### Feature importance SHAP : SHapley Additive exPlanations
alternative to permutation feature importance
gbr svr only

In [ ]:
# SVR LR
# vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()

explainer_LinearSVR12 = shap.Explainer(LinearSVR, x12_train, feature_names=x12_train.columns)
shap_values12_LinearSVR = explainer_LinearSVR1(x12_test)

In [ ]:
shap.plots.beeswarm(shap_values12_LinearSVR)

In [ ]:
shap.summary_plot(shap_values12_LinearSVR,features=x12_train, feature_names=x12_train.columns,plot_type='bar')

In [ ]:
# GBR
explainer_GBR12 = shap.Explainer(GBR, x12_train, feature_names=x12_train.columns)
shap_values_GBR12 = explainer_GBR12(x12_test)
shap.plots.beeswarm(shap_values_GBR12)


In [ ]:
shap.summary_plot(shap_values_GBR12,features=x12_test, feature_names=x12_test.columns,plot_type='bar')

### CONCLUSION RUN 1 
- log n ameliore pas trop mon model
- ajoueter des features par pas de bon resultats
comment : chercher dans analyse bivarie, si une qui 
correl bien avec GHE.
un bon candidat : third type.

## Model RUN 2 : Add New feature (cf analyse bivarie)
### Prepa data 

In [ ]:
list_new_feature2 = ["LogLargestPropertyUseTypeGFA","LogSecondLargestPropertyUseTypeGFA","LogThirdLargestPropertyUseTypeGFA"]

In [ ]:
data2[list_new_feature2].isnull().sum()

In [ ]:
# replace Nan by 0
data2[list_new_feature2].fillna(0)

In [ ]:
list_col_struct_model2 = [
 'CouncilDistrictCode',
 'Classe_Age',
 'NumberofBuildings',
 'NumberofFloors', 
 "LogLargestPropertyUseTypeGFA","LogSecondLargestPropertyUseTypeGFA","LogThirdLargestPropertyUseTypeGFA","GHGEmissionsIntensity"]

In [ ]:
df_list_tempo2 = data2[list_col_struct_model2]

In [ ]:
df_list_tempo2.shape

In [ ]:
df_list_tempo2.isna().sum()

In [ ]:
df_list_tempo2.dropna(subset ="GHGEmissionsIntensity",inplace = True)

In [ ]:
df_list_tempo2.reset_index(inplace = True)
df_list_tempo2.drop(columns = ["index"],inplace =True)

In [ ]:
df_list_tempo2 = df_list_tempo2.fillna(0)

In [ ]:
df_list_tempo2

In [ ]:
# Create a dataframe
#df_model1 = pd.concat([data2[list_tempo],encoder_df], axis=1)
df_model2 = df_list_tempo2.join(encoder_df)

In [ ]:
df_model2

In [ ]:
X2 = df_model2.drop (columns = ["GHGEmissionsIntensity"])

In [ ]:
X2

In [ ]:
X2.isna().count()

In [ ]:
Y.head()

#### Normalistation des datas

In [ ]:
# Another option MinMaxScaler : transform features by scaling each feature to a given range.
# Min-max is a scaling technique where values are rescaled and shifted so that they range between 0 and 1
# scalerMinMax = preprocessing.MinMaxScaler() #default=(0, 1)
names2 = X2.columns
d2 = scalerMinMax.fit_transform(X2)
x2_normMM = pd.DataFrame(d2, columns=names2)
x2_normMM.head() 

In [ ]:
# normaliser tout entre 0 et 1 mieux que juste variance 
# Trop de varaibilite entre les features, plus visible, plus de perf pour la modelisation car plus petite valeur
norm2 = sklearn.preprocessing.normalize(X2, norm='l2', axis=0, copy=True, return_norm=False) # axis = 0 means normalization amoung the col/features..better
x2_norm = pd.DataFrame(norm2, columns = X2.columns)
x2_norm 

### Creer un jeu de données test et train : Test train split

In [ ]:
x2_train, x2_test = train_test_split(x2_norm, test_size = 0.2, random_state = 1) #random state : ontrols how the data is shuffled before the split is implemented.

In [ ]:
x2_train.shape, x2_test.shape, y_train.shape, y_test.shape

In [ ]:
# testons si nos echantillons sont bien equilibres:
(((Y.value_counts())/Y.shape[0])*100).round(2)

### Test Model :
#### Regression lineaire Simple

#####  Regression lineaire classique : baseline de regression des moindres carres

In [ ]:
# On entraîne ce modèle sur les données d'entrainement
lr.fit(x2_train,y_train) # fit trouver les param qui minimise les moindres carres pour avoir le meilleur coef
print(lr.coef_) # 1 coef par feature X

In [ ]:
# Save coef in a dataframe
df2_coef_lr =pd.DataFrame(lr.coef_, X2.columns, columns = ['Coeff'])
df2_coef_lr #High coef, means High impact for the prediction
#output : the highest coef are Number of Building and area, make sens for CO2 emission...

In [ ]:
# Resultats
print(lr.intercept_) # see the intercept (b or Beta0) of the regression equation

In [ ]:
y_pred2_lr_test = lr.predict(x2_test)
y_pred2_lr_train = lr.predict(x2_train)

In [ ]:
# On récupère l'erreur de norme 2 sur le jeu de données test comme baseline
baseline_error_test2 = np.mean((y_pred2_lr_test - y_test) ** 2) # moy des erreurs quadratiques entre donnee et prediction sur test, cad diff entre valeur obs et valeur predites 
baseline_error_train2 = np.mean((y_pred2_lr_train - y_train) ** 2) # moy des erreurs quadratiques entre donnee et prediction sur training, cad diff entre valeur obs et valeur predites 

# Erreur quadratique = MSE: Mean Scare Error
print("Pour la régression linéaire basique : La MSE2 du training set est de : " + str(baseline_error_train2))
print("Pour la régression linéaire basique : La MSE2 du testing set est de : " + str(baseline_error_test2))

In [ ]:
# MSE via Sklearn
MSE_lr_test2_sk = sklearn.metrics.mean_squared_error(y_test, y_pred2_lr_test)
MSE_lr_train2_sk = sklearn.metrics.mean_squared_error(y_train, y_pred2_lr_train)

In [ ]:
MSE_lr_test2_sk, MSE_lr_train2_sk # Meme resultats

In [ ]:
# RMSE via Sklearn : Racine de MSE, vire le carre
RMSE_lr_test2_sk = sklearn.metrics.mean_squared_error(y_test, y_pred2_lr_test, squared= False)
RMSE_lr_train2_sk = sklearn.metrics.mean_squared_error(y_train, y_pred2_lr_train, squared= False)

In [ ]:
RMSE_lr_test2_sk, RMSE_lr_train2_sk 

In [ ]:
# Visualisation perf du model2
plt.scatter(y_train, y_pred2_lr_train)
plt.xlabel("Emission Co2 Prédites sur training set")
plt.ylabel("Emission Co2 réelles sur training set")


In [ ]:
# Visualisation 
plt.scatter(y_test, y_pred2_lr_test)
plt.xlabel("Emission Co2 Prédites sur jeu test")
plt.ylabel("Emission Co2 réelles sur jeu test")
# pas mieux sur testing set

In [ ]:
# Another way to visually assess the performance of our model is to plot its residuals = the difference between the actual y-array values and the predicted y-array values.
plt.hist(y_test - y_pred2_lr_test) #distribution difference between obs and prediction
# output : not normally distributed : not good :)

##### Dummy Regressor

In [ ]:
from sklearn import dummy
dum = dummy.DummyRegressor(strategy='mean')

In [ ]:
# Define Model
model2_dum = dum.fit(x2_train, y_train)
print("Dummy-R2 sur training set est de (0): "+str(model2_dum.score(x2_train,y_train))+" : OK.")# output= 0

# Prédiction sur le jeu de test
y_pred2_dum = model2_dum.predict(x2_test)
print("Dummy-R2 sur testing set est de : "+str(model2_dum.score(x2_test,y_test)))

In [ ]:
# Evaluate
print("MSE2 with Dummy Regressor : {:.4f}".format(sklearn.metrics.mean_squared_error(y_test, y_pred2_dum)) )
print("RMSE2 with Dummy Regressor : {:.4f}".format(np.sqrt(sklearn.metrics.mean_squared_error(y_test, y_pred2_dum)) ))

In [ ]:
model2_dum.n_features_in_

In [ ]:
# Visualize res (actual testing values vs predicted)
plt.scatter(y_test, y_pred2_dum)
plt.xlabel("Emission Co2 Prédites par Dummy")
plt.ylabel("Emission Co2 réelles")
plt.title("DummyRegressor method on testing set")


##### Ameliorer la regression lineraire : Trouver le meilleur alpha/lambda as hyperparametres :
ridge
lasso
gridsearch sklearn

###### Ridge/Tykhonov regularisation : permet de reduire l amplitude des coef d une regression lineaire et eviter le surapprentissage
pose une contrainte : lambda/ alpha

In [ ]:
# Regression sous contrainte lambda, test pour chaque lambda/alpha
coefs22 = []
errors22 = []
for a in alphas:
    ridge.set_params(alpha=a)
    ridge.fit(x2_train, y_train)
    coefs22.append(ridge.coef_)
    errors22.append([baseline_error_test2, np.mean((ridge.predict(x2_test) - y_test) ** 2)])

In [ ]:
# MSE
MSE2_lr_ridge = np.mean((ridge.predict(x2_test) - y_test) ** 2)
print("Pour la régression linéaire ridge : La MSE du testing set est de : " + str(MSE2_lr_ridge.round(2)))

In [ ]:
# Visualisation de la difference entre (erreur quadratique : cest diff entre yobs et y predit (f(x)) au carree)
erreurs22= np.array(errors22)
plt.semilogx(alphas,erreurs22[:,1]) # valeur de erreur en fonction de alpha
plt.semilogx(alphas, erreurs22[:,0]) # baseline error, from linear regression (erreur quadratrique moy sur jeu test), ev
# output : il existe une valeur alpha/lambda qui ameliore la prediction (ameliore l erreur sur les données de test) 
# ici tous les alphas ameliorent

In [ ]:
np.argmin(erreurs22[:,1]) # on trouve l indice de l erreur mini qui ameliore l erreur

In [ ]:
# on veut retrouver alpha pour l erreur min
alpha_opt2 = alphas[np.argmin(erreurs22[:,1])]
alpha_opt2

In [ ]:
# Chemin de regularisation
plt.semilogx(alphas, coefs22)
plt.semilogx([alpha_opt2, alpha_opt2], [-4,4], 'r--')

In [ ]:
coef_opt22 = coefs22[np.argmin(erreurs22[:,1])]
coef_opt22

##### Lasso : methode de selection de variables et de reduction de dimension supervise
remove celle norme 1 de B sup a t
region admissible +^- valeur absolue B

In [ ]:
from sklearn.linear_model import Lasso
# Regression sous contrainte lambda, test pour chaque lambda/alpha
#lasso = linear_model.Lasso()

coefs222 = []
errors222 = []
for a in alphas:
    lasso.set_params(alpha=a)
    lasso.fit(x2_train, y_train)
    coefs222.append(lasso.coef_)
    errors222.append([baseline_error_test2, np.mean((lasso.predict(x2_test) - y_test) ** 2)])

In [ ]:
# MSE
MSE2_lr_lasso = np.mean((lasso.predict(x2_test) - y_test) ** 2)
print("Pour la régression linéaire lasso : La MSE du testing set est de : " + str(MSE2_lr_lasso.round(2)))


In [ ]:
# Visualisation de la difference entre (erreur quadratique : cest diff entre yobs et y predit (f(x)) au carree)
erreurs222= np.array(errors222)
plt.semilogx(alphas,erreurs222[:,1]) # valeur de erreur en fonction de alpha
plt.semilogx(alphas, erreurs222[:,0]) # baseline error, from linear regression (erreur quadratrique moy sur jeu test), ev
# output : il existe une valeur alpha/lambda qui ameliore la prediction (ameliore l erreur sur les données de test) 

In [ ]:
alpha_opt2_lasso = alphas[np.argmin(erreurs222[:,1])]

# Chemin de regularisation
plt.semilogx(alphas, coefs222)
plt.semilogx([alpha_opt2_lasso, alpha_opt2_lasso], [-4,4], 'r--')

In [ ]:
# Chemin de regularisation
plt.semilogx(alphas, coefs222)
plt.xlim(.0001,.1)
plt.ylim(-.4,.2)
plt.semilogx([alpha_opt2_lasso, alpha_opt2_lasso], [-.5,.5], 'r--')

In [ ]:
alpha_opt2_lasso 

In [ ]:
np.argmin(erreurs22[:,1])

In [ ]:
coef_opt2 = coefs222[np.argmin(erreurs222[:,1])]
coef_opt2

##### Grid search technique . recherche hyperparametre
test les models
GridSearchCV should be used to find the optimal parameters to train your final model.
Typically, you should run GridSearchCV
then look at the parameters that gave the model with the best score.
You should then take these parameters and train your final model on all of the data. 
It is important to note that if you have trained your final model on all of your data, you cannot test it. 

###### LR

In [ ]:
grid.fit(x2_train, y_train)

In [ ]:

print("MSE gridsearch with kf=10 and lr method: %.2f"
              % np.mean((grid.predict(x2_test) - y_test) ** 2))
print ("r2 / variance : ", grid.best_score_)

###### Lasso

In [ ]:
grid_lasso.fit(x2_train, y_train)

In [ ]:
# Results
print("MSE gridsearch with kf=10 and lasso method: %.2f"
              % np.mean((grid_lasso.predict(x2_test) - y_test) ** 2))
print ("r2 / variance : ", grid_lasso.best_score_)
print('Best Hyperparameters: %s' % grid_lasso.best_params_)

###### Ridge

In [ ]:
grid_ridge.fit(x2_train, y_train)

In [ ]:
# Results
print("MSE gridsearch with kf=10 and lasso method: %.2f"
              % np.mean((grid_ridge.predict(x2_test) - y_test) ** 2))
print ("r2 / variance : ", grid_ridge.best_score_)
print('Best Hyperparameters: %s' % grid_ridge.best_params_)

##### Validation croisee pour choisir k fold
= méthode d’estimation de fiabilité d’un modèle fondée sur une technique d’échantillonnage.
choisir les meilleurs hyper-paramètres
but : optimiser l’erreur quadratique moyenne (Mean Squared Error, MSE)

###### Cross Val score : MSE

In [ ]:
# How to estimate the accuray of a linear kernel (10 times)
# return array of score (default = R2) of the estimator for each run
scores2_lr = cross_val_score(lr, X2, Y, cv=kf,scoring = make_scorer(mean_squared_error)) 
scores2_lr # DONNE SCORE (MSE) A CHAQUE ITERATION


In [ ]:
print ("Le meilleur MSE, validation croisée LR : " + str ((min(scores2_lr)).round(2)) )

In [ ]:
# Rigde
scores2_ridge = cross_val_score(ridge, X2, Y, cv=kf,scoring = make_scorer(mean_squared_error)) 
print ("Le meilleur MSE, validation croisée LR2 : " + str ((min(scores2_ridge)).round(2)) )

In [ ]:
# Lasso
scores2_lasso = cross_val_score(lasso, X2, Y, cv=kf,scoring = make_scorer(mean_squared_error) ) 
print ("Le meilleur MSE, validation croisée LR2 : " + str ((min(scores2_lasso)).round(2)) )

In [ ]:
# report performance
print('MSE from lr, mean (sdt) : %.3f (%.3f)' % (np.mean(scores2_lr), np.std(scores2_lr)))

print('MSE from ridge, mean (sdt):  %.3f (%.3f)' % (np.mean(scores2_ridge), np.std(scores2_ridge)))

print('MSE from lasso, mean (sdt):  %.3f (%.3f)' % (np.mean(scores2_lasso), np.std(scores2_lasso)))

#### Methode à Noyau = transfo non lineaire en lineaire

##### Linear SVR (Support Vector Machine for linear regression)

###### SVR libsvm implementation 

In [ ]:
#lr_SVR2 = make_pipeline(SVR(C=1.0, epsilon=0.2)) #StandardScaler()
#lr_SVR = sklearn.svm.SVR(kernel='linear')
# Entraine le model
lr_SVR.fit(x2_train, y_train)

In [ ]:
# Prediction sur testing set
y_pred2_SVR = lr_SVR.predict(x2_test)

In [ ]:
# MSE via Sklearn
MSE2_SVR = sklearn.metrics.mean_squared_error(y_test, y_pred2_SVR)
print ("le coef de correlation de la SVR sur testing set est de : MSE_SVR = "+ str(MSE2_SVR.round(2)))


In [ ]:
coef2_SVR = lr_SVR.coef_
coef2_SVR

###### SVR liblinear  implementation 

In [ ]:
# Entraine le model
LinearSVR.fit(x2_train, y_train)

In [ ]:
# Prediction sur testing set
y_pred2_LinearSVR = LinearSVR.predict(x2_test)

In [ ]:
# MSE via Sklearn
MSE2_LinearSVR = sklearn.metrics.mean_squared_error(y_test, y_pred2_LinearSVR)
print ("MSE_LinearSVR = "+ str(MSE2_LinearSVR.round(2)))


In [ ]:
coef2_LinearSVR = LinearSVR.coef_
coef2_LinearSVR

##### kRR Regression ridge kernel  : RGF /Gaussien
identical to support SVR
KRR uses squared error loss while support vector regression uses -insensitive loss, both combined with l2 regularization

In [ ]:
# entraîner le classifieur sur le jeu d'entrainement
kRR_GRF.fit(x2_train, y_train)


In [ ]:
# prédire sur le jeu de test
y_pred2_kRR = kRR_GRF.predict(x2_test)

In [ ]:
# calculer la MSE sur le jeu de test
MSE2_kRR_GRF = sklearn.metrics.mean_squared_error(y_test, y_pred2_kRR)
print ("MSE_kRR_RBF : {:.2f}".format(MSE2_kRR_GRF))


##### Polynomial-kRR

In [ ]:

                                     
# entraîner sur le jeu d'entrainement
kRR_poly1.fit(x2_train, y_train)

# prédire sur le jeu de test
y_pred2_kRR_poly1 = kRR_poly1.predict(x2_test)


# calculer la MSE sur le jeu de test
MSE2_kRR_poly1 = sklearn.metrics.mean_squared_error(y_test, y_pred2_kRR_poly1)
print ("MSE2_kRR_poly1 : {:.4f}".format(MSE2_kRR_poly1))

In [ ]:
                                   
# entraîner sur le jeu d'entrainement
kRR_poly2.fit(x2_train, y_train)

# prédire sur le jeu de test
y_pred2_kRR_poly2 = kRR_poly2.predict(x2_test)


# calculer la MSE sur le jeu de test
MSE2_kRR_poly2 = sklearn.metrics.mean_squared_error(y_test, y_pred2_kRR_poly2)
print ("MSE_kRR_poly2 : {:.4f}".format(MSE2_kRR_poly2))

In [ ]:
                                   
# entraîner sur le jeu d'entrainement
kRR_poly3.fit(x2_train, y_train)

# prédire sur le jeu de test
y_pred2_kRR_poly3 = kRR_poly3.predict(x2_test)


# calculer la MSE sur le jeu de test
MSE2_kRR_poly3 = sklearn.metrics.mean_squared_error(y_test, y_pred2_kRR_poly3)
print ("MSE_kRR_poly3 : {:.4f}".format(MSE2_kRR_poly3))

In [ ]:

MSE2_krr_poly_deg=[]
for deg in deglist:
    kRR_poly522 = sklearn.kernel_ridge.KernelRidge(
        alpha=100, # valeur par défaut 
        kernel='poly', # noyau polynomial de degre 1
        gamma=0.01,
        degree=deg)                                 
    # entraîner sur le jeu d'entrainement
    kRR_poly522.fit(x2_train, y_train)
    # prédire sur le jeu de test
    y_pred2_kRR_poly5 = kRR_poly522.predict(x2_test)
    # calculer la MSE sur le jeu de test
    MSE2_krr_poly_deg.append(sklearn.metrics.mean_squared_error(y_test, y_pred2_kRR_poly5))

plt.plot(deglist, MSE2_krr_poly_deg, 'ob')
plt.ylim([4.25, 4.28])
plt.xlabel('degree du polynome')
plt.ylabel('MSE2 kernel polynomial')

#### METHODE ENSEMBLISTE
##### GDRegressor : GBR

In [ ]:
#sklearn.linear_model.SGDRegressor(loss='squared_error', *, penalty='l2', alpha=0.0001, l1_ratio=0.15, fit_intercept=True, max_iter=1000, tol=0.001, shuffle=True, verbose=0, epsilon=0.1, random_state=None, learning_rate='invscaling', eta0=0.01, power_t=0.25, early_stopping=False, validation_fraction=0.1, n_iter_no_change=5, warm_start=False, average=False)
# GBR = sklearn.ensemble.GradientBoostingRegressor(random_state=0) #default criteirm friedman mse , mse upgraded
GBR.fit(x2_train, y_train)

In [ ]:
GBR_pred2 = GBR.predict(x2_test)

In [ ]:
GBR.score(x2_test,y_test) #R2

In [ ]:
# MSE via Sklearn
MSE2_GBR = sklearn.metrics.mean_squared_error(y_test, GBR_pred2)
print ("MSE_GBR : {:.2f}".format(MSE2_GBR))

In [ ]:
#FEATURE IMPORTANCE
GBR.feature_importances_

In [ ]:
# Gradient boosting train 
#crossvalidation=KFold(n_splits=10,shuffle=True,random_state=1)
#search_grid={'n_estimators':[100],'learning_rate':[.001,0.01,.1],'max_depth':[1,2,4],'subsample':[.5,.75,1],'random_state':[1]}
#search=GridSearchCV(estimator=GBR,param_grid=search_grid,scoring='neg_mean_squared_error',n_jobs=1,cv=crossvalidation)

search.fit(x2_train,y_train)



In [ ]:
search.best_score_, search.best_params_

In [ ]:
scores_GBR2 = cross_val_score(search, x2_test,y_test, cv=kf,scoring = make_scorer(mean_squared_error)) 


In [ ]:
scores_GBR2,np.mean(scores_GBR2), np.min(scores_GBR2)

In [ ]:
GBR_Pred_grid2 = search.predict(x2_test)
MSE2_GBR_grid = sklearn.metrics.mean_squared_error(y_test, GBR_Pred_grid2)
print ("MSE2_GBR-grid : {:.2f}".format(MSE2_GBR_grid))

##### RandomForest

In [ ]:
# Etablissement Model
# RFR = RandomForestRegressor(max_depth=2, random_state=0)

# Entrainement
RFR.fit(x2_train, y_train)

# Predicton
RFR_Pred2 = RFR.predict(x2_test)

In [ ]:
RFR.apply(x2_test) #return leaf indice

In [ ]:
RFR.feature_importances_

In [ ]:
# MSE via Sklearn
MSE2_RFR = sklearn.metrics.mean_squared_error(y_test, RFR_Pred2)
print ("MSE2_RFR : {:.2f}".format(MSE2_RFR))

###### gridsearch

In [ ]:
# RFR
grid_search_rf.fit(x2_train, y_train)

In [ ]:
grid_search_rf.best_score_, grid_search_rf.best_params_

In [ ]:
scores_RFR2 = cross_val_score(rf, x2_test,y_test, cv=crossvalidation,scoring = make_scorer(mean_squared_error)) 
scores_RFR2,np.mean(scores_RFR2), np.min(scores_RFR2)

In [ ]:
RFR_Pred2_grid = grid_search_rf.predict(x2_test)
MSE2_RFR_grid = sklearn.metrics.mean_squared_error(y_test, RFR_Pred2_grid)
print ("MSE2_RFR-grid: {:.2f}".format(MSE2_RFR_grid))

### Feature importance SHAP : SHapley Additive exPlanations
alternative to permutation feature importance
gbr svr only

In [ ]:
# SVR LR
# vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()

explainer2_LinearSVR = shap.Explainer(LinearSVR, x2_train, feature_names=x2_train.columns)
shap_values_LinearSVR2 = explainer2_LinearSVR(x2_test)


In [ ]:
shap.plots.beeswarm(shap_values_LinearSVR2)

In [ ]:
shap.summary_plot(shap_values_LinearSVR2,features=x2_test, feature_names=x2_test.columns,plot_type='bar')

In [ ]:
# GBR

explainer_GBR2 = shap.Explainer(GBR, x2_train, feature_names=x2_train.columns)
shap_values_GBR2 = explainer_GBR2(x2_test)
shap.plots.beeswarm(shap_values_GBR2)


In [ ]:
shap.summary_plot(shap_values_GBR2,features=x2_test, feature_names=x2_test.columns,plot_type='bar')

COONCLUSION RUN 2 type 3
- ameliore un peu
depend des model
-meilleur GBR
- meilleur feature importance :
third largest Property USe type
classe hotel
Number of building

## Model RUN 3 star score
### Prepa data : Add New feature : starScore

In [ ]:
data2['ENERGYSTARScore'].isnull().sum()

In [ ]:
data2_short = data2.dropna(subset ='ENERGYSTARScore')
data2_short['ENERGYSTARScore'].isnull().sum()
data2_short.shape

In [ ]:
# replace Nan by 0
data2_short[list_new_feature2].fillna(0)

In [ ]:
list_col_struct_model3 = [
 'CouncilDistrictCode',
 'Classe_Age',
 'NumberofBuildings',
 'NumberofFloors', 'ENERGYSTARScore',
 "LogLargestPropertyUseTypeGFA","LogSecondLargestPropertyUseTypeGFA","LogThirdLargestPropertyUseTypeGFA","GHGEmissionsIntensity"]

In [ ]:
df_list_tempo3 = data2_short[list_col_struct_model3]

In [ ]:
df_list_tempo3.shape

In [ ]:
df_list_tempo3.isna().sum()

In [ ]:
df_list_tempo3.dropna(subset ="GHGEmissionsIntensity",inplace = True)

In [ ]:
df_list_tempo3.reset_index(inplace = True)
df_list_tempo3.drop(columns = ["index"],inplace =True)

In [ ]:
df_list_tempo3 = df_list_tempo3.fillna(0)

In [ ]:
df_list_tempo3

In [ ]:
# Create a dataframe

df_model3 = df_list_tempo3.join(encoder_df)

In [ ]:
df_model3

In [ ]:
X3 = df_model3.drop (columns = ["GHGEmissionsIntensity"])
Y3 = df_model3["GHGEmissionsIntensity"]

In [ ]:
X3.shape,Y3.shape

In [ ]:
X3.isna().count()

#### Normalistation des datas

In [ ]:
# Another option MinMaxScaler : transform features by scaling each feature to a given range.
# Min-max is a scaling technique where values are rescaled and shifted so that they range between 0 and 1
# scalerMinMax = preprocessing.MinMaxScaler() #default=(0, 1)
names3 = X3.columns
d3 = scalerMinMax.fit_transform(X3)
x3_normMM = pd.DataFrame(d3, columns=names3)
x3_normMM.head() 

In [ ]:
# normaliser tout entre 0 et 1 mieux que juste variance 
# Trop de varaibilite entre les features, plus visible, plus de perf pour la modelisation car plus petite valeur
norm3 = sklearn.preprocessing.normalize(X3, norm='l2', axis=0, copy=True, return_norm=False) # axis = 0 means normalization amoung the col/features..better
x3_norm = pd.DataFrame(norm3, columns = X3.columns)
x3_norm 

### Creer un jeu de données test et train : Test train split

In [ ]:
x3_train, x3_test, y3_train, y3_test = train_test_split(x3_norm,Y3, test_size = 0.2, random_state = 1) #random state : ontrols how the data is shuffled before the split is implemented.

In [ ]:
x3_train.shape, x3_test.shape, y3_train.shape, y3_test.shape

In [ ]:
# testons si nos echantillons sont bien equilibres:
(((Y3.value_counts())/Y3.shape[0])*100).round(2)

### Test Model :
#### Regression lineaire Simple

#####  Regression lineaire classique : baseline de regression des moindres carres

In [ ]:
# On entraîne ce modèle sur les données d'entrainement
lr.fit(x3_train,y3_train) # fit trouver les param qui minimise les moindres carres pour avoir le meilleur coef
print(lr.coef_) # 1 coef par feature X

In [ ]:
# Save coef in a dataframe
df3_coef_lr =pd.DataFrame(lr.coef_, X3.columns, columns = ['Coeff'])
df3_coef_lr #High coef, means High impact for the prediction
#output : the highest coef are Number of Building and area, make sens for CO2 emission...

In [ ]:
# Resultats
print(lr.intercept_) # see the intercept (b or Beta0) of the regression equation

In [ ]:
y_pred3_lr_test = lr.predict(x3_test)
y_pred3_lr_train = lr.predict(x3_train)

In [ ]:
# On récupère l'erreur de norme 2 sur le jeu de données test comme baseline
baseline_error_test3 = np.mean((y_pred3_lr_test - y3_test) ** 2) # moy des erreurs quadratiques entre donnee et prediction sur test, cad diff entre valeur obs et valeur predites 
baseline_error_train3 = np.mean((y_pred3_lr_train - y3_train) ** 2) # moy des erreurs quadratiques entre donnee et prediction sur training, cad diff entre valeur obs et valeur predites 

# Erreur quadratique = MSE: Mean Scare Error
print("Pour la régression linéaire basique : La MSE3 du training set est de : " + str(baseline_error_train3))
print("Pour la régression linéaire basique : La MSE3 du testing set est de : " + str(baseline_error_test3))

In [ ]:
# MSE via Sklearn
MSE_lr_test3_sk = sklearn.metrics.mean_squared_error(y3_test, y_pred3_lr_test)
MSE_lr_train3_sk = sklearn.metrics.mean_squared_error(y3_train, y_pred3_lr_train)

In [ ]:
MSE_lr_test3_sk, MSE_lr_train3_sk # Meme resultats

In [ ]:
# RMSE via Sklearn : Racine de MSE, vire le carre
RMSE_lr_test3_sk = sklearn.metrics.mean_squared_error(y3_test, y_pred3_lr_test, squared= False)
RMSE_lr_train3_sk = sklearn.metrics.mean_squared_error(y3_train, y_pred3_lr_train, squared= False)

In [ ]:
RMSE_lr_test3_sk, RMSE_lr_train3_sk 

In [ ]:
# Visualisation perf du model3
plt.scatter(y3_train, y_pred3_lr_train)
plt.xlabel("Emission Co2 Prédites sur training set")
plt.ylabel("Emission Co2 réelles sur training set")

# OUTPUT trop parfait...pb

In [ ]:
# Visualisation 
plt.scatter(y3_test, y_pred3_lr_test)
plt.xlabel("Emission Co2 Prédites sur jeu test")
plt.ylabel("Emission Co2 réelles sur jeu test")
# pas mieux sur testing set

##### Dummy Regressor

In [ ]:
from sklearn import dummy
dum = dummy.DummyRegressor(strategy='mean')

In [ ]:
# Define Model
model3_dum = dum.fit(x3_train, y3_train)
print("Dummy-R2 sur training set est de (0): "+str(model3_dum.score(x3_train,y3_train))+" : OK.")# output= 0

# Prédiction sur le jeu de test
y_pred3_dum = model3_dum.predict(x3_test)
print("Dummy-R2 sur testing set est de : "+str(model3_dum.score(x3_test,y3_test)))

In [ ]:
# Evaluate
print("MSE3 with Dummy Regressor : {:.4f}".format(sklearn.metrics.mean_squared_error(y3_test, y_pred3_dum)) )
print("RMSE3 with Dummy Regressor : {:.4f}".format(np.sqrt(sklearn.metrics.mean_squared_error(y3_test, y_pred3_dum)) ))

##### Ameliorer la regression lineraire : Trouver le meilleur alpha/lambda as hyperparametres :
ridge
lasso
gridsearch sklearn

###### Ridge/Tykhonov regularisation : permet de reduire l amplitude des coef d une regression lineaire et eviter le surapprentissage
pose une contrainte : lambda/ alpha

In [ ]:
# Regression sous contrainte lambda, test pour chaque lambda/alpha
coefs3 = []
errors3 = []
for a in alphas:
    ridge.set_params(alpha=a)
    ridge.fit(x3_train, y3_train)
    coefs3.append(ridge.coef_)
    errors3.append([baseline_error_test3, np.mean((ridge.predict(x3_test) - y3_test) ** 2)])

In [ ]:
# MSE
MSE3_lr_ridge = np.mean((ridge.predict(x3_test) - y3_test) ** 2)
print("Pour la régression linéaire ridge : La MSE du testing set est de : " + str(MSE3_lr_ridge.round(2)))

In [ ]:
# Visualisation de la difference entre (erreur quadratique : cest diff entre yobs et y predit (f(x)) au carree)
erreurs3= np.array(errors3)
plt.semilogx(alphas,erreurs3[:,1]) # valeur de erreur en fonction de alpha
plt.semilogx(alphas, erreurs3[:,0]) # baseline error, from linear regression (erreur quadratrique moy sur jeu test), ev
# output : il existe une valeur alpha/lambda qui ameliore la prediction (ameliore l erreur sur les données de test) 


In [ ]:
np.argmin(erreurs3[:,1]) # on trouve l indice de l erreur mini qui ameliore l erreur

In [ ]:
# on veut retrouver alpha pour l erreur min
alpha_opt3 = alphas[np.argmin(erreurs3[:,1])]
alpha_opt3

In [ ]:
# Chemin de regularisation
plt.semilogx(alphas, coefs3)
plt.semilogx([alpha_opt3, alpha_opt3], [-4,4], 'r--')

In [ ]:
coef_opt3 = coefs3[np.argmin(erreurs3[:,1])]
coef_opt3

##### Lasso : methode de selection de variables et de reduction de dimension supervise
remove celle norme 1 de B sup a t
region admissible +^- valeur absolue B

In [ ]:
from sklearn.linear_model import Lasso
# Regression sous contrainte lambda, test pour chaque lambda/alpha
#lasso = linear_model.Lasso()

coefs3las = []
errors3las = []
for a in alphas:
    lasso.set_params(alpha=a)
    lasso.fit(x3_train, y3_train)
    coefs3las.append(lasso.coef_)
    errors3las.append([baseline_error_test3, np.mean((lasso.predict(x3_test) - y3_test) ** 2)])

In [ ]:
# MSE
MSE3_lr_lasso = np.mean((lasso.predict(x3_test) - y3_test) ** 2)
print("Pour la régression linéaire lasso : La MSE du testing set est de : " + str(MSE3_lr_lasso.round(2)))


In [ ]:
# Visualisation de la difference entre (erreur quadratique : cest diff entre yobs et y predit (f(x)) au carree)
erreurs3las= np.array(errors3las)
plt.semilogx(alphas,erreurs3las[:,1]) # valeur de erreur en fonction de alpha
plt.semilogx(alphas, erreurs3las[:,0]) # baseline error, from linear regression (erreur quadratrique moy sur jeu test), ev
# output : il existe une valeur alpha/lambda qui ameliore la prediction (ameliore l erreur sur les données de test) 

In [ ]:
alpha_opt3_lasso = alphas[np.argmin(erreurs3las[:,1])]

# Chemin de regularisation
plt.semilogx(alphas, coefs3las)
plt.semilogx([alpha_opt3_lasso, alpha_opt3_lasso], [-4,4], 'r--')

In [ ]:
# Chemin de regularisation
plt.semilogx(alphas, coefs3las)
plt.xlim(.0001,.1)
plt.ylim(-.4,.2)
plt.semilogx([alpha_opt3_lasso, alpha_opt3_lasso], [-.5,.5], 'r--')

In [ ]:
alpha_opt3_lasso 

In [ ]:
np.argmin(erreurs3las[:,1])

In [ ]:
coef_opt3las = coefs3las[np.argmin(erreurs3las[:,1])]
coef_opt3las

##### Grid search technique . recherche hyperparametre


###### LR

In [ ]:
grid.fit(x3_train, y3_train)

In [ ]:

print("MSE gridsearch with kf=10 and lr method: %.2f"
              % np.mean((grid.predict(x3_test) - y3_test) ** 2))
print ("r2 / variance : ", grid.best_score_)

###### Lasso

In [ ]:
grid_lasso.fit(x3_train, y3_train)

In [ ]:
# Results
print("MSE gridsearch with kf=10 and lasso method: %.2f"
              % np.mean((grid_lasso.predict(x3_test) - y3_test) ** 2))
print ("r2 / variance : ", grid_lasso.best_score_)
print('Best Hyperparameters: %s' % grid_lasso.best_params_)

###### Ridge

In [ ]:
grid_ridge.fit(x3_train, y3_train)

In [ ]:
# Results
print("MSE gridsearch with kf=10 and ridge method: %.2f"
              % np.mean((grid_ridge.predict(x3_test) - y3_test) ** 2))
print ("r2 / variance : ", grid_ridge.best_score_)
print('Best Hyperparameters: %s' % grid_ridge.best_params_)

##### Validation croisee pour choisir k fold
= méthode d’estimation de fiabilité d’un modèle fondée sur une technique d’échantillonnage.
choisir les meilleurs hyper-paramètres
but : optimiser l’erreur quadratique moyenne (Mean Squared Error, MSE)

###### Cross Val score : MSE

In [ ]:
# How to estimate the accuray of a linear kernel (10 times)
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import make_scorer, r2_score
# return array of score (default = R2) of the estimator for each run
scores3_lr = cross_val_score(lr, X3, Y3, cv=kf,scoring = make_scorer(mean_squared_error)) 
scores3_lr # DONNE SCORE (MSE) A CHAQUE ITERATION


In [ ]:
print ("Le meilleur MSE, validation croisée LR3 : " + str ((min(scores3_lr)).round(2)) )

In [ ]:
# Rigde
scores3_ridge = cross_val_score(ridge, X3, Y3, cv=kf,scoring = make_scorer(mean_squared_error)) 
print ("Le meilleur MSE, validation croisée Ridge3 : " + str ((min(scores3_ridge)).round(2)) )

In [ ]:
# Lasso
scores3_lasso = cross_val_score(lasso, X3, Y3, cv=kf,scoring = make_scorer(mean_squared_error) ) 
print ("Le meilleur MSE, validation croisée Lasso 3 : " + str ((min(scores3_lasso)).round(2)) )

In [ ]:
# report performance
print('MSE from lr, mean (sdt) : %.3f (%.3f)' % (np.mean(scores3_lr), np.std(scores3_lr)))

print('MSE from ridge, mean (sdt):  %.3f (%.3f)' % (np.mean(scores3_ridge), np.std(scores3_ridge)))

print('MSE from lasso, mean (sdt):  %.3f (%.3f)' % (np.mean(scores3_lasso), np.std(scores3_lasso)))

#### Test Autre Model: Methode à Noyau =
transfo non lineaire en lineaire

##### Linear SVR (Support Vector Machine for linear regression)

###### SVR libsvm implementation 

In [ ]:
# Entraine le model
lr_SVR.fit(x3_train, y3_train)

In [ ]:
# Prediction sur testing set
y_pred3_SVR = lr_SVR.predict(x3_test)

In [ ]:
# MSE via Sklearn
MSE3_SVR = sklearn.metrics.mean_squared_error(y3_test, y_pred3_SVR)
print ("le coef de correlation de la SVR sur testing set est de : MSE3_SVR = "+ str(MSE3_SVR.round(2)))


In [ ]:
coef3_SVR = lr_SVR.coef_
coef3_SVR

###### SVR liblinear  implementation 

In [ ]:
# Entraine le model
LinearSVR.fit(x3_train, y3_train)

In [ ]:
# Prediction sur testing set
y_pred3_LinearSVR = LinearSVR.predict(x3_test)

In [ ]:
# MSE via Sklearn
MSE3_LinearSVR = sklearn.metrics.mean_squared_error(y3_test, y_pred3_LinearSVR)
print ("MSE_LinearSVR = "+ str(MSE3_LinearSVR.round(2)))


In [ ]:
coef3_LinearSVR = LinearSVR.coef_
coef3_LinearSVR

##### kRR Regression ridge kernel trics : RGF /Gaussien
identical to support SVR
KRR uses squared error loss while support vector regression uses -insensitive loss, both combined with l2 regularization

In [ ]:
# entraîner le classifieur sur le jeu d'entrainement
kRR_GRF.fit(x3_train, y3_train)


In [ ]:
# prédire sur le jeu de test
y_pred3_kRR = kRR_GRF.predict(x3_test)

In [ ]:
# calculer la MSE sur le jeu de test
MSE3_kRR_GRF = sklearn.metrics.mean_squared_error(y3_test, y_pred3_kRR)
print ("MSE_kRR_RBF : {:.2f}".format(MSE3_kRR_GRF))


##### Polynomial-kRR

In [ ]:
# entraîner sur le jeu d'entrainement
kRR_poly1.fit(x3_train, y3_train)

# prédire sur le jeu de test
y_pred3_kRR_poly1 = kRR_poly1.predict(x3_test)


# calculer la MSE sur le jeu de test
MSE3_kRR_poly1 = sklearn.metrics.mean_squared_error(y3_test, y_pred3_kRR_poly1)
print ("MSE3_kRR_poly1 : {:.4f}".format(MSE3_kRR_poly1))

In [ ]:
# entraîner sur le jeu d'entrainement
kRR_poly2.fit(x3_train, y3_train)

# prédire sur le jeu de test
y_pred3_kRR_poly2 = kRR_poly2.predict(x3_test)


# calculer la MSE sur le jeu de test
MSE3_kRR_poly2 = sklearn.metrics.mean_squared_error(y3_test, y_pred3_kRR_poly2)
print ("MSE3_kRR_poly2 : {:.4f}".format(MSE3_kRR_poly2))

In [ ]:
# entraîner sur le jeu d'entrainement
kRR_poly3.fit(x3_train, y3_train)

# prédire sur le jeu de test
y_pred3_kRR_poly3 = kRR_poly3.predict(x3_test)


# calculer la MSE sur le jeu de test
MSE3_kRR_poly3 = sklearn.metrics.mean_squared_error(y3_test, y_pred3_kRR_poly3)
print ("MSE_kRR_poly3 : {:.4f}".format(MSE3_kRR_poly3))

In [ ]:

MSE3_krr_poly_deg=[]
for deg in deglist:
    kRR_poly53 = sklearn.kernel_ridge.KernelRidge(
        alpha=100, # valeur par défaut 
        kernel='poly', # noyau polynomial de degre 1
        gamma=0.01,
        degree=deg)                                 
    # entraîner sur le jeu d'entrainement
    kRR_poly53.fit(x3_train, y3_train)
    # prédire sur le jeu de test
    y_pred3_kRR_poly5 = kRR_poly53.predict(x3_test)
    # calculer la MSE sur le jeu de test
    MSE3_krr_poly_deg.append(sklearn.metrics.mean_squared_error(y3_test, y_pred3_kRR_poly5))

plt.plot(deglist, MSE3_krr_poly_deg, 'ob')

plt.xlabel('degree du polynome')
plt.ylabel('MSE2 kernel polynomial')

In [ ]:
min(MSE3_krr_poly_deg)

#### METHODE ENSEMBLISTE
##### GDRegressor : GBR

In [ ]:
#sklearn.linear_model.SGDRegressor(loss='squared_error', *, penalty='l2', alpha=0.0001, l1_ratio=0.15, fit_intercept=True, max_iter=1000, tol=0.001, shuffle=True, verbose=0, epsilon=0.1, random_state=None, learning_rate='invscaling', eta0=0.01, power_t=0.25, early_stopping=False, validation_fraction=0.1, n_iter_no_change=5, warm_start=False, average=False)
# GBR = sklearn.ensemble.GradientBoostingRegressor(random_state=0) #default criteirm friedman mse , mse upgraded
GBR.fit(x3_train, y3_train)

In [ ]:
GBR_pred3 = GBR.predict(x3_test)

In [ ]:
GBR.score(x3_test,y3_test) #R2

In [ ]:
# MSE via Sklearn
MSE3_GBR = sklearn.metrics.mean_squared_error(y3_test, GBR_pred3)
print ("MSE3_GBR : {:.2f}".format(MSE3_GBR))

In [ ]:
#FEATURE IMPORTANCE
GBR.feature_importances_

In [ ]:
# Gradient boosting train 
#crossvalidation=KFold(n_splits=10,shuffle=True,random_state=1)
#search_grid={'n_estimators':[100],'learning_rate':[.001,0.01,.1],'max_depth':[1,2,4],'subsample':[.5,.75,1],'random_state':[1]}
#search=GridSearchCV(estimator=GBR,param_grid=search_grid,scoring='neg_mean_squared_error',n_jobs=1,cv=crossvalidation)

search.fit(x3_test,y3_test)



In [ ]:
search.best_score_, search.best_params_

In [ ]:
scores_GBR3 = cross_val_score(search, x3_test,y3_test, cv=kf,scoring = make_scorer(mean_squared_error)) 


In [ ]:
scores_GBR3,np.mean(scores_GBR3), np.min(scores_GBR3)

In [ ]:
GBR_Pred3_grid = search.predict(x3_test)
MSE3_GBR_grid = sklearn.metrics.mean_squared_error(y3_test, GBR_Pred3_grid)
print ("MSE3_GBR-grid : {:.2f}".format(MSE3_GBR_grid))

##### RandomForest

In [ ]:
# Entrainement
RFR.fit(x3_train, y3_train)

# Predicton
RFR_Pred3 = RFR.predict(x3_test)

In [ ]:
RFR.apply(x3_test) #return leaf indice

In [ ]:
RFR.feature_importances_

In [ ]:
# MSE via Sklearn
MSE3_RFR = sklearn.metrics.mean_squared_error(y3_test, RFR_Pred3)
print ("MSE3_RFR : {:.2f}".format(MSE3_RFR))

###### gridsearch

In [ ]:
# RFR
grid_search_rf.fit(x3_train, y3_train)

In [ ]:
grid_search_rf.best_score_, grid_search_rf.best_params_

In [ ]:
scores_RFR3 = cross_val_score(rf, x3_train, y3_train, cv=crossvalidation,scoring = make_scorer(mean_squared_error)) 
scores_RFR3,np.mean(scores_RFR3), np.min(scores_RFR3)

In [ ]:
RFR_Pred3_grid = grid_search_rf.predict(x3_test)
MSE3_RFR_grid = sklearn.metrics.mean_squared_error(y3_test, RFR_Pred3_grid)
print ("MSE2_RFR-grid: {:.2f}".format(MSE3_RFR_grid))

### Feature importance SHAP : SHapley Additive exPlanations
alternative to permutation feature importance
gbr svr only

In [ ]:
import shap
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# SVR LR
# vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()

explainer3_LinearSVR = shap.Explainer(LinearSVR, x3_train, feature_names=x3_train.columns)
shap_values3_LinearSVR = explainer3_LinearSVR(x3_test)


In [ ]:
shap.plots.beeswarm(shap_values3_LinearSVR)

In [ ]:
shap.summary_plot(shap_values3_LinearSVR,features=x3_train, feature_names=x3_train.columns,plot_type='bar')

In [ ]:
# GBR
explainer_GBR3 = shap.Explainer(GBR, x3_train, feature_names=x3_train.columns)
shap_values_GBR3 = explainer_GBR3(x3_test)
shap.plots.beeswarm(shap_values_GBR3)

In [ ]:
shap.summary_plot(shap_values_GBR3,features=x3_test, feature_names=x3_test.columns,plot_type='bar')

### Conclusion : 
- best score MSE with GBR 
- Energie Star Score ameliore beaucoup le model : on garde pour le model Energie

In [ ]:
# Export data
data2_short.to_csv("data2_short.csv",index =False)
data2_short.shape